# S6E9 · V8：Strong XGBoost Feature Views + Expanded Probit Ensemble

你刚才 V7 的实际结果：

- V5 baseline OOF：`0.946098745`
- V7 nested hill OOF：约 `0.946141`
- V7 rules experimental OOF：约 `0.946152`
- V7 Public LB：**`0.94619`**

这说明：**meta-ensemble 方法有效，但当前基础模型本身太相似、也不够强。**

V8 不继续微调 hill-climb 权重，而是先强化最弱的一块：XGBoost。

公开 S6E9 结果里已经有：

- 单 XGBoost Public 约 `0.94640`
- 有公开评论报告：相同特征下 10-fold XGB CV 约 `0.94623`、LB 约 `0.94640`
- heuljax 公开报告自己最好 10-fold XGB CV 约 `0.94652`

而我们旧 XGB OOF 只有：

`0.9459315`

所以当前最合理的判断不是“马上把整个项目换成神经网络”，而是：

> **先把 XGB recipe 拉到公开强单模的合理区间；如果仍卡住，再切到 RealMLP / Transformer 这种真正低相关模型。**

V8 会训练两个新的 XGB feature views：

1. **strong_rich_te**
   - 保留你现有 digit + interactions + fold-local frequency + nested triple TE；
   - 换成更强的 lossguide / regularized XGB 参数。

2. **strong_digit_freq**
   - digit + interactions + fold-local frequency；
   - **不做 target encoding**；
   - 目的不是只追单模 AUC，而是得到和 rich TE 不同的排序。

训练完后，把两个新 engine 加回：

`probit → nested hill climb → logistic stack`

重新选择最终提交。

如果这两个新 XGB 都无法把 OOF 明显推过 `0.9462`，V9 才值得切到 RealMLP / Transformer。


## 1. 加载原有训练与编码函数

下面保留你 V3 的算法：原来的行内数字特征、LightGBM 内层五折目标编码、相同外层五折和参数；仅把新训练目录前缀改为 `v4_lightgbm_`。函数定义不会开始训练。

目标编码仍严格在外层训练折内部计算；验证/测试只使用外层训练折的映射。不要为了省时间在整个训练集上先算一份目标编码。

原训练器保存模型、每折预测与编码来源的训练行索引；**模型文件本身不包含全部特征/目标编码过程**，以后推理必须配合相同代码与训练数据重建映射。

In [1]:
"""S6E9: fixed-fold CatBoost; optional LightGBM with cross-fitted target encoding.

Default: python s6e9_train_v3.py
Optional: python s6e9_train_v3.py --models catboost lightgbm
Dependencies: numpy pandas scikit-learn catboost; optional lightgbm.
No training occurs when this module is imported.
"""
import argparse
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]


def features(frame, model_name):
    """Row-local features only: no labels or full-data target statistics."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),
                                 ("Daily_Commute_km", 10, 3),
                                 ("Charging_Stations_Near_Home", 1, 2),
                                 ("Charging_Stations_Near_Work", 1, 2)]:
        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)
    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)
    x["income_last_2"] = income % 100
    x["income_last_3"] = income % 1000
    if model_name == "catboost":
        for col in NUMERIC:
            scale = 10 if col == "Daily_Commute_km" else 1
            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    else:
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000
    return x


def encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):
    """Inner OOF for training TE; outer-training-only maps for validation/test."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y)
    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))
    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]
    for col in keys:
        encoded = np.full(len(a), np.nan)
        for fit_idx, hold_idx in inner:
            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})
            stats = subset.groupby("key")["y"].agg(["sum", "count"])
            prior = labels[fit_idx].mean()
            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()
        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])
        prior = labels.mean()
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        a[f"{col}_te"] = encoded
        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()
        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()
        assert np.isfinite(encoded).all()
    # Label-free frequency: exclude a training row's own count contribution.
    counts = a["Annual_Income_USD"].value_counts()
    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)
    for frame in (b, c):
        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)
    for col in cats:
        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))
        for frame in (a, b, c):
            frame[col] = frame[col].astype(dtype)
    return a, b, c


def run(project="/Users/yw/unsw/Kaggle", models=("catboost",), threads=-1):
    models = tuple(models)
    if not models or len(set(models)) != len(models) or not set(models) <= {"catboost", "lightgbm"}:
        raise ValueError("models must contain distinct choices from catboost/lightgbm")
    # Check requested dependencies before starting any expensive training.
    if "catboost" in models:
        from catboost import CatBoostClassifier
    if "lightgbm" in models:
        try:
            import lightgbm as lgb
        except ImportError as error:
            raise ImportError("Optional LightGBM is unavailable; install lightgbm or run CatBoost only.") from error

    project = Path(project)
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    assert train.drop(columns=TARGET).columns.tolist() == test.columns.tolist()
    assert train.id.is_unique and test.id.is_unique and not train.id.isin(test.id).any()
    assert not train.isna().any().any() and not test.isna().any().any()
    assert set(train[TARGET].unique()) == {"No", "Yes"}
    for frame in (train, test):
        assert np.allclose(frame.Annual_Income_USD, frame.Annual_Income_USD.round())
        assert np.allclose(frame.Daily_Commute_km * 10, (frame.Daily_Commute_km * 10).round())
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    out = project / "outputs" / ("v4_lightgbm_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    (out / "config.json").write_text(json.dumps({
        "models": models, "folds": 5, "seed": 42, "threads": threads,
        "train_rows": len(train), "test_rows": len(test),
        "note": "LightGBM/encoding and blend are unvalidated candidates; compare their OOF results.",
    }, indent=2))
    print("Output directory:", out, flush=True)
    all_oof, all_test, records = {}, {}, []

    def save_submission(name, probabilities):
        assert len(probabilities) == len(test)
        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()
        pd.DataFrame({"id": test.id.to_numpy(), TARGET: probabilities}).to_csv(
            out / f"submission_{name}.csv", index=False)

    for name in models:
        x, xt = features(train, name), features(test, name)
        assert x.columns.tolist() == xt.columns.tolist()
        cats = x.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        oof = np.full(len(train), np.nan)
        coverage = np.zeros(len(train), dtype=np.int8)
        test_sum = np.zeros(len(test))
        for fold, (ti, vi) in enumerate(folds, start=1):
            print(f"\n========== {name} | Fold {fold}/5 ==========", flush=True)
            a, b, c = x.iloc[ti], x.iloc[vi], xt
            if name == "catboost":
                # Same features/parameters as the successful single-fold candidate.
                model = CatBoostClassifier(
                    iterations=2000, learning_rate=0.03, depth=7,
                    loss_function="Logloss", eval_metric="AUC", random_seed=41 + fold,
                    thread_count=threads, allow_writing_files=False, verbose=400,
                )
                model.fit(a, y.iloc[ti], cat_features=cats, eval_set=(b, y.iloc[vi]),
                          early_stopping_rounds=150, use_best_model=True)
                model.save_model(str(out / f"{name}_fold{fold}.cbm"))
                best = int(model.get_best_iteration()) + 1
            else:
                a, b, c = encode_fold(a, y.iloc[ti], b, c, seed=1041 + fold)
                bins = [1023 if col == "Annual_Income_USD" else 255 for col in a.columns]
                model = lgb.LGBMClassifier(
                    n_estimators=5000, learning_rate=0.05, max_depth=5, num_leaves=31,
                    min_child_samples=100, reg_lambda=5.0, max_bin_by_feature=bins,
                    objective="binary", metric="auc", random_state=41 + fold, n_jobs=threads,
                    verbosity=-1, deterministic=True, force_col_wise=True,
                )
                model.fit(a, y.iloc[ti], eval_set=[(b, y.iloc[vi])], eval_metric="auc",
                          callbacks=[lgb.early_stopping(200, first_metric_only=True), lgb.log_evaluation(400)])
                model.booster_.save_model(str(out / f"{name}_fold{fold}.txt"))
                best = int(model.best_iteration_)
                # Persist fold-specific TE maps' source indices, for reproducible later inference.
                np.save(out / f"{name}_fold{fold}_train_indices.npy", ti)
            assert np.array_equal(model.classes_, [0, 1])
            vp = model.predict_proba(b)[:, 1]
            tp = model.predict_proba(c)[:, 1]
            assert np.isfinite(vp).all() and np.isfinite(tp).all()
            oof[vi], coverage[vi] = vp, coverage[vi] + 1
            test_sum += tp
            auc = float(roc_auc_score(y.iloc[vi], vp))
            records.append({"model": name, "fold": fold, "auc": auc, "best_trees": best})
            np.savez_compressed(out / f"{name}_fold{fold}_predictions.npz",
                                valid_indices=vi, valid_ids=train.id.iloc[vi].to_numpy(),
                                valid_prediction=vp, test_ids=test.id.to_numpy(), test_prediction=tp)
            pd.DataFrame(records).to_csv(out / "fold_scores.csv", index=False)
            print("Fold AUC:", auc, flush=True)
        assert (coverage == 1).all() and np.isfinite(oof).all()
        all_oof[name], all_test[name] = oof, test_sum / len(folds)
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": oof}).to_csv(
            out / f"oof_{name}.csv", index=False)
        save_submission(name, all_test[name])
        print(name, "OOF AUC:", roc_auc_score(y, oof), flush=True)

    if len(models) == 2:
        # One predeclared blend, avoiding an extensive weight search on the same OOF.
        all_oof["blend_50_50"] = (all_oof["catboost"] + all_oof["lightgbm"]) / 2
        all_test["blend_50_50"] = (all_test["catboost"] + all_test["lightgbm"]) / 2
        save_submission("blend_50_50", all_test["blend_50_50"])
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(),
                      "prediction": all_oof["blend_50_50"]}).to_csv(out / "oof_blend_50_50.csv", index=False)
    scores = {name: float(roc_auc_score(y, prediction)) for name, prediction in all_oof.items()}
    best_name = max(scores, key=scores.get)
    save_submission("best_oof", all_test[best_name])
    (out / "summary.json").write_text(json.dumps({
        "oof_auc": scores, "selected": best_name,
        "selection_note": "Selected on development OOF; this is not an independent audit or a Kaggle score.",
    }, indent=2))
    print("\nOOF comparison:", scores)
    print("Selected:", best_name, "\nCandidate:", out / "submission_best_oof.csv")
    return out




## 2. 路径与旧结果核对工具

默认路径来自你的已保存日志。旧结果缺失、ID/标签不一致、折划分不同、CSV 不等于每折预测的汇总，都会先报错，不会悄悄混用结果。

这能核对旧运行的标签、预测和划分，但不能追溯证明旧原始特征从未被修改：V3 没有保存原 CSV 的哈希。请保留当时原始的 `train.csv` / `test.csv`；本版本会记录其哈希供后续复用核对。

In [2]:
import sys
import hashlib
import importlib.metadata


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def input_hashes(project):
    root = Path(project) / "data"
    names = ["train.csv", "test.csv", "sample_submission.csv"]
    return {name: file_hash(root / name) for name in names if (root / name).is_file()}


def probabilities(values, expected_length, description):
    p = np.asarray(values, dtype=float)
    require(p.shape == (expected_length,), f"{description}: wrong prediction shape")
    require(np.isfinite(p).all(), f"{description}: NaN or infinity")
    require(((p >= 0) & (p <= 1)).all(), f"{description}: probabilities outside [0, 1]")
    return p


def align_csv(path, ids, columns):
    """Do not blend CSVs by their incidental row order: align explicitly by id."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"缺少文件：{path}。请先确认原 outputs 目录；尚未开始新训练。")
    frame = pd.read_csv(path)
    require(set(["id", *columns]).issubset(frame.columns), f"{path.name}: missing columns")
    require(frame.id.notna().all() and frame.id.is_unique, f"{path.name}: invalid/duplicate ids")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted) and wanted.isin(frame.id).all(), f"{path.name}: id sets differ")
    return frame.set_index("id").loc[wanted, columns].reset_index()


def load_project(project):
    project = Path(project)
    for name in ("train.csv", "test.csv"):
        if not (project / "data" / name).is_file():
            raise FileNotFoundError(f"找不到 {project / 'data' / name}；请检查 PROJECT。")
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    require(TARGET in train, "train.csv is missing the target")
    require(train.drop(columns=TARGET).columns.tolist() == test.columns.tolist(), "Train/test schema differs")
    require(train.id.is_unique and test.id.is_unique, "Duplicate id")
    require(not train.id.isin(test.id).any(), "Train/test ids overlap")
    require(not train.isna().any().any() and not test.isna().any().any(), "Missing values found")
    require(set(train[TARGET].unique()) == {"No", "Yes"}, "Expected No/Yes labels")
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    require(y.value_counts().min() >= 10, "Too few rows per class for nested five-fold validation")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    sample = project / "data/sample_submission.csv"
    if sample.exists():
        sample_frame = pd.read_csv(sample)
        require(sample_frame.columns.tolist() == ["id", TARGET], "Unexpected submission columns")
        require(sample_frame.id.is_unique and len(sample_frame) == len(test), "Invalid sample submission ids")
        require(pd.Index(test.id).isin(sample_frame.id).all(), "Sample submission has different ids")
    return train, test, y, folds


def read_run(run_dir, model_name, train, test, y, folds, expected_auc=None):
    """Verify saved OOF, test averages, and the actual five validation splits."""
    run_dir = Path(run_dir)
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"找不到 {config_path}；请把 BASELINE_DIR 指向原训练结果目录。")
    config = json.loads(config_path.read_text())
    require(config.get("folds") == 5 and config.get("seed") == 42, "Run does not use V3 folds/seed")
    require(model_name in config.get("models", []), f"This run does not contain {model_name}")
    require(config.get("train_rows") == len(train) and config.get("test_rows") == len(test), "Dataset sizes differ")
    oof_frame = align_csv(run_dir / f"oof_{model_name}.csv", train.id, ["target", "prediction"])
    test_frame = align_csv(run_dir / f"submission_{model_name}.csv", test.id, [TARGET])
    require(np.array_equal(oof_frame.target.to_numpy(), y.to_numpy()), "OOF labels do not match current train.csv")
    oof = probabilities(oof_frame.prediction, len(train), "OOF")
    test_pred = probabilities(test_frame[TARGET], len(test), "Test")
    reconstructed = np.full(len(train), np.nan)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test))
    fold_records = []
    for fold, (_, vi) in enumerate(folds, start=1):
        path = run_dir / f"{model_name}_fold{fold}_predictions.npz"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；无法核对原五折，不会静默混用其他划分。")
        with np.load(path, allow_pickle=False) as saved:
            require(np.array_equal(saved["valid_indices"], vi), f"Fold {fold}: split differs")
            require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), f"Fold {fold}: train order/ids differ")
            require(np.array_equal(saved["test_ids"], test.id), f"Fold {fold}: test order/ids differ")
            vp = probabilities(saved["valid_prediction"], len(vi), f"Fold {fold} OOF")
            tp = probabilities(saved["test_prediction"], len(test), f"Fold {fold} test")
            reconstructed[vi] = vp
            coverage[vi] += 1
            test_sum += tp
            fold_records.append({"fold": fold, "auc": float(roc_auc_score(y.iloc[vi], vp))})
    require((coverage == 1).all(), "OOF coverage is not exactly once per row")
    require(np.allclose(reconstructed, oof, atol=1e-12, rtol=0), "OOF CSV differs from fold predictions")
    require(np.allclose(test_sum / 5, test_pred, atol=1e-12, rtol=0), "Test CSV is not this run's five-fold average")
    score = float(roc_auc_score(y, oof))
    if expected_auc is not None:
        require(abs(score - expected_auc) < 1e-8,
                f"旧 OOF AUC 是 {score:.10f}，不是本次预期的 {expected_auc:.10f}；请检查目录版本。")
    return {"oof": oof, "test": test_pred, "auc": score, "fold_scores": pd.DataFrame(fold_records)}


def compare_and_save(project, baseline_dir, lightgbm_dir, train, test, y, folds, old, new, hashes):
    """Four predeclared candidates only; this is development selection, not an audit."""
    require(input_hashes(project) == hashes, "Input CSV files changed during this session; stop and inspect")
    out = Path(project) / "outputs" / ("v4_compare_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    # Include the old baseline so a weaker LightGBM does not automatically replace it.
    candidates = [("catboost_baseline", 1.0), ("lightgbm", 0.0),
                  ("blend_cat75_lgb25", 0.75), ("blend_cat50_lgb50", 0.50)]
    records, per_fold, candidate_test = [], [], {}
    base_fold = [roc_auc_score(y.iloc[vi], old["oof"][vi]) for _, vi in folds]
    for name, weight in candidates:
        op = weight * old["oof"] + (1 - weight) * new["oof"]
        tp = probabilities(weight * old["test"] + (1 - weight) * new["test"], len(test), name)
        auc = float(roc_auc_score(y, op))
        deltas = []
        for fold, (_, vi) in enumerate(folds, start=1):
            fa = float(roc_auc_score(y.iloc[vi], op[vi]))
            delta = float(fa - base_fold[fold - 1])
            deltas.append(delta)
            per_fold.append({"candidate": name, "fold": fold, "auc": fa, "delta_vs_catboost": delta})
        records.append({"candidate": name, "catboost_weight": weight,
                        "oof_auc": auc, "delta_vs_catboost": auc - old["auc"],
                        "improved_folds": sum(d > 0 for d in deltas),
                        "worst_fold_delta": min(deltas)})
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": op}).to_csv(
            out / f"oof_{name}.csv", index=False)
        submission = pd.DataFrame({"id": test.id.to_numpy(), TARGET: tp})
        sample_path = Path(project) / "data/sample_submission.csv"
        if sample_path.exists():
            sample_ids = pd.read_csv(sample_path, usecols=["id"])
            submission = sample_ids.merge(submission, on="id", how="left", validate="one_to_one", sort=False)
        probabilities(submission[TARGET], len(test), "Submission")
        submission.to_csv(out / f"submission_{name}.csv", index=False)
        candidate_test[name] = submission
    # max keeps the first candidate on an exact tie (the old baseline).
    best = max(records, key=lambda row: row["oof_auc"])
    comparison = pd.DataFrame(records).sort_values("oof_auc", ascending=False, kind="stable").reset_index(drop=True)
    comparison.to_csv(out / "comparison.csv", index=False)
    pd.DataFrame(per_fold).to_csv(out / "comparison_by_fold.csv", index=False)
    candidate_test[best["candidate"]].to_csv(out / "submission_next_candidate.csv", index=False)
    (out / "continuation_summary.json").write_text(json.dumps({
        "baseline_dir": str(baseline_dir), "lightgbm_dir": str(lightgbm_dir),
        "baseline_oof_auc": old["auc"], "user_reported_baseline_public_auc": 0.94570,
        "selected_by_development_oof": best, "candidates": records,
        "input_sha256": hashes,
        "note": "OOF was used for early stopping and model/weight selection; not an independent test. No new Kaggle score is known.",
    }, indent=2), encoding="utf-8")
    print(comparison.to_string(index=False, float_format=lambda v: f"{v:.8f}"))
    print("\n比较目录：", out)
    print("OOF 选择的候选：", best["candidate"])
    print("候选 CSV：", out / "submission_next_candidate.csv")
    if best["candidate"] == "catboost_baseline":
        print("本轮候选均未超过旧 CatBoost。保留原 0.94570 提交，不必为了新版本而重新提交。")
    else:
        print("这是开发 OOF 最优候选，不保证 Public/Private 分数更高。请同时检查逐折变化，不做大规模权重搜索。")
    return out, comparison


In [3]:
PROJECT = Path("/Users/yw/unsw/Kaggle")
BASELINE_DIR = PROJECT / "outputs/v3_20260925_180457_278867"
EXPECTED_BASELINE_AUC = 0.9456182325266642
THREADS = -1  # 保持原设定。需要同时使用电脑时可改成 4。

# 这份 notebook 已经完整跑过 LightGBM，因此默认直接复用已保存目录，避免 Run All 重训。
# 这里只复用【完整】结果，不支持某一折训练到一半的自动恢复。
LGB_RESULT_DIR = PROJECT / "outputs/v4_lightgbm_20260930_163526_079778"
# 示例（请用程序实际打印的目录，不要直接复制占位符）：
# LGB_RESULT_DIR = PROJECT / "outputs/v4_lightgbm_实际时间戳"


## 3. 先检查，不训练

正常情况下应打印旧 OOF AUC 约 `0.9456182325`。若报错缺少文件，请调整 `PROJECT` / `BASELINE_DIR` 或恢复旧输出，不要随便换成另一次训练目录。

`sample_submission.csv` 存在时会检查它并用其 ID 顺序导出；不存在时保留原 `test.csv` 的 ID 顺序。

In [4]:
# 这一格只检查，不训练。
try:
    import lightgbm as lgb
except (ImportError, OSError) as error:
    raise RuntimeError(
        "当前 Jupyter 内核无法导入 lightgbm。请先在这个内核安装/修复后重启内核；"
        "缺 Python 包时可在单独一格运行 %pip install lightgbm。"
    ) from error

train, test, y, folds = load_project(PROJECT)
old = read_run(BASELINE_DIR, "catboost", train, test, y, folds,
               expected_auc=EXPECTED_BASELINE_AUC)
DATA_HASHES = input_hashes(PROJECT)
print("当前 Python：", sys.executable)
print("旧结果检查通过；CatBoost OOF AUC：", old["auc"])
print(old["fold_scores"].to_string(index=False))
print("本轮不会重训 CatBoost。接下来运行第 4 节才会训练 LightGBM。")


当前 Python： /opt/miniconda3/envs/dl/bin/python
旧结果检查通过；CatBoost OOF AUC： 0.9456182325266642
 fold      auc
    1 0.944681
    2 0.945343
    3 0.946578
    4 0.945892
    5 0.945626
本轮不会重训 CatBoost。接下来运行第 4 节才会训练 LightGBM。


## 4. 只补跑 LightGBM 五折

参数保持原候选：5000 轮上限、学习率 0.05、深度 5、最多 31 个叶节点、连续 200 轮无提升早停。每折的最佳轮次由验证集决定。

第一次运行 `LGB_RESULT_DIR = None`。完整训练完成后程序会在同一内核记住目录；重启内核后，可以将第 2 节 `LGB_RESULT_DIR` 改成实际打印的目录，跳过训练。**中断在某一折时，原训练器没有自动断点续训；重新启动会新建目录并从五折的第一折重跑。**

不要因为这一节打印 `Selected: lightgbm` 就认为它比旧模型强：它在这一节是唯一参选模型，最终比较在下一节。

In [5]:
# 只有这里会启动训练；最多新训练一套 LightGBM 五折。
if LGB_RESULT_DIR is None:
    require(input_hashes(PROJECT) == DATA_HASHES, "检查后数据文件已改变")
    lightgbm_dir = run(project=PROJECT, models=("lightgbm",), threads=THREADS)
    require(input_hashes(PROJECT) == DATA_HASHES, "训练过程中数据文件已改变")
    (lightgbm_dir / "data_fingerprints.json").write_text(json.dumps(DATA_HASHES, indent=2))
    versions = {name: importlib.metadata.version(name)
                for name in ("numpy", "pandas", "scikit-learn", "lightgbm")}
    (lightgbm_dir / "package_versions.json").write_text(json.dumps(versions, indent=2))
    LGB_RESULT_DIR = lightgbm_dir  # 同一内核再次运行此格会复用完整结果。
else:
    lightgbm_dir = Path(LGB_RESULT_DIR)
    require(lightgbm_dir.resolve() != BASELINE_DIR.resolve(), "不能把旧 CatBoost 目录当成 LightGBM 目录")
    fingerprint_path = lightgbm_dir / "data_fingerprints.json"
    if not fingerprint_path.is_file():
        raise FileNotFoundError(f"缺少 {fingerprint_path}。请选择本 notebook 已完成的 LightGBM 目录。")
    require(json.loads(fingerprint_path.read_text()) == DATA_HASHES, "复用目录对应的数据文件版本不同")
    print("复用完整的 LightGBM 结果：", lightgbm_dir)

new = read_run(lightgbm_dir, "lightgbm", train, test, y, folds)
print("LightGBM OOF AUC：", new["auc"])
print("注意：此训练目录里的 submission_best_oof.csv 只在 LightGBM 本轮内选择。")
print("请继续运行第 5 节，才能与旧 CatBoost 一起比较。")


复用完整的 LightGBM 结果： /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778
LightGBM OOF AUC： 0.9459463810009869
注意：此训练目录里的 submission_best_oof.csv 只在 LightGBM 本轮内选择。
请继续运行第 5 节，才能与旧 CatBoost 一起比较。


## 5. 用相同 OOF 比较四个候选，导出提交文件

只比较预先规定的四个方案，不做密集权重搜索。75/25 表示 `0.75 × CatBoost + 0.25 × LightGBM`；50/50 同理。

先按 ID 对齐预测，然后计算整体 OOF AUC，并记录逐折相对旧 CatBoost 的变化。`improved_folds` 是五个验证折中有几个提高；它**不是显著性检验**。OOF 同时用于早停和选择，因此这些结果是开发分数，不是独立评估。

新目录 `v4_compare_时间戳` 中：`comparison.csv` 是总体比较，`comparison_by_fold.csv` 是逐折变化，`submission_next_candidate.csv` 是开发 OOF 选出的候选。旧模型也参与选择；新方案没超过旧模型时保留旧模型，不需要重复提交同一批预测。

In [6]:
comparison_dir, comparison = compare_and_save(
    PROJECT, BASELINE_DIR, lightgbm_dir, train, test, y, folds, old, new, DATA_HASHES
)
comparison


        candidate  catboost_weight    oof_auc  delta_vs_catboost  improved_folds  worst_fold_delta
blend_cat50_lgb50       0.50000000 0.94598624         0.00036801               5        0.00032103
         lightgbm       0.00000000 0.94594638         0.00032815               5        0.00024666
blend_cat75_lgb25       0.75000000 0.94585286         0.00023463               5        0.00020918
catboost_baseline       1.00000000 0.94561823         0.00000000               0        0.00000000

比较目录： /Users/yw/unsw/Kaggle/outputs/v4_compare_20260930_213722_827091
OOF 选择的候选： blend_cat50_lgb50
候选 CSV： /Users/yw/unsw/Kaggle/outputs/v4_compare_20260930_213722_827091/submission_next_candidate.csv
这是开发 OOF 最优候选，不保证 Public/Private 分数更高。请同时检查逐折变化，不做大规模权重搜索。


,candidate,catboost_weight,oof_auc,delta_vs_catboost,improved_folds,worst_fold_delta
0,blend_cat50_lgb50,0.50,0.945986,0.000368,5,0.000321
1,lightgbm,0.00,0.945946,0.000328,5,0.000247
2,blend_cat75_lgb25,0.75,0.945853,0.000235,5,0.000209
3,catboost_baseline,1.00,0.945618,0.000000,0,0.000000


## 怎么读结果、怎么决定下一步

以旧 OOF **0.9456182325** 为比较基线，而不是把它与 Public **0.94570** 跨数据集直接比输赢。即使新 OOF 更高，也不保证新的 Public 或 Private 更高。很小的提升尤其要看逐折变化；不要为了追逐 Public 的偶然变化反复调权重。

若 LightGBM 或融合没有提高，本轮仍然得到了有效的负结果：保持原方案。下一项可单独测试把 CatBoost 迭代上限从 2000 提到 3500，并保持特征、学习率和相同五折；这需要新训练，**本 notebook 没有启动该实验**。

参考：
- [竞赛评分与截止时间](https://www.kaggle.com/competitions/playground-series-s6e9/overview)
- [scikit-learn：ROC AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
- [scikit-learn：目标编码的内部交叉拟合](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_target_encoder_cross_val.html)
- [LightGBM：early_stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html)

算法主体延续用户上传的 V3 实现；新增部分负责旧运行核对、复用、ID 对齐、数据版本记录及四候选比较。

## 6. V5 参数：10-fold XGBoost

先跑这一节，不要急着改很多参数。

这次最重要的变化不是“换一组 XGB 参数”，而是：

1. **5-fold → 10-fold**；
2. 保留 digit decomposition；
3. 每个 outer fold 内做 frequency encoding；
4. 每个 outer fold 内做 **inner 5-fold 的 Triple Target Encoding**；
5. XGBoost 只看到训练折产生的编码，validation/test 不参与 target statistics；
6. 最后同时比较 raw probability blend 和 **rank blend**。

`XGB_RESULT_DIR = None` 时训练；完整训练结束后，把打印出来的目录填回去即可复用。


In [7]:
# ===== V5 configuration =====
CURRENT_PUBLIC_SCORE = 0.94618

XGB_SPLITS = 10
XGB_SEED = 42

# Triple TE：公开 S6E9 XGB 方案里常见的多平滑 target encoding。
TE_SMOOTHINGS = (1.0, 5.0, 20.0)
INNER_TE_SPLITS = 5

# 复用你刚才已经跑完的 10-fold XGBoost，避免 V6 再训练一次。
XGB_RESULT_DIR = PROJECT / "outputs/v5_xgb10_20260930_205349_440980"
# 示例：
# XGB_RESULT_DIR = PROJECT / "outputs/v5_xgb10_20260930_xxxxxx"

# Apple Silicon / CPU 直接用 hist。不要在 Mac 上硬设 CUDA。
XGB_PARAMS = dict(
    n_estimators=4000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=20,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_lambda=8.0,
    reg_alpha=0.03,
    max_bin=512,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    early_stopping_rounds=200,
    n_jobs=THREADS,
)

print("Current reported Public:", CURRENT_PUBLIC_SCORE)
print("XGB folds:", XGB_SPLITS)
print("Triple-TE smoothings:", TE_SMOOTHINGS)


Current reported Public: 0.94618
XGB folds: 10
Triple-TE smoothings: (1.0, 5.0, 20.0)


## 7. XGBoost 特征：保留 exact-value / digit signal，但严格防 leakage

这一版特征分成三类：

- **row-local**：digit、income suffix/prefix、charging total/gap、income ratios、subsidy interactions；
- **frequency**：只使用当前 outer-training fold 的频数；
- **Triple TE**：outer-training fold 内再做 inner 5-fold OOF 编码。

特别注意：Target Encoding 不能先在完整 `train.csv` 上算好再切 10 folds。那样 CV 会虚高。


In [8]:
import gc

def add_xgb_row_features(frame):
    """Label-free row features. Safe to build before CV."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()

    digit_specs = {
        "Age": (1, 2),
        "Annual_Income_USD": (1, 6),
        "Daily_Commute_km": (10, 3),
        "Number_of_Cars_Owned": (1, 1),
        "Charging_Stations_Near_Home": (1, 2),
        "Charging_Stations_Near_Work": (1, 2),
        "Environmental_Concern_Level": (1, 1),
    }

    for col, (scale, positions) in digit_specs.items():
        if col not in frame.columns:
            continue
        values = (
            pd.to_numeric(frame[col], errors="coerce")
            .fillna(0)
            .to_numpy()
        )
        values = np.rint(values * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (
                np.abs(values) // (10 ** digit) % 10
            ).astype(np.int8)

    if "Annual_Income_USD" in frame.columns:
        income = np.rint(
            pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
        ).astype(np.int64)
        x["income_last_2"] = income % 100
        x["income_last_3"] = income % 1000
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000

    if {"Charging_Stations_Near_Home", "Charging_Stations_Near_Work"} <= set(frame.columns):
        home = pd.to_numeric(frame["Charging_Stations_Near_Home"], errors="coerce").fillna(0)
        work = pd.to_numeric(frame["Charging_Stations_Near_Work"], errors="coerce").fillna(0)
        x["charging_total"] = home + work
        x["charging_gap"] = home - work

    if {"Annual_Income_USD", "Number_of_Cars_Owned"} <= set(frame.columns):
        income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
        cars = pd.to_numeric(frame["Number_of_Cars_Owned"], errors="coerce").fillna(0)
        x["income_per_car"] = income / (cars + 1.0)

    if {"Annual_Income_USD", "Daily_Commute_km"} <= set(frame.columns):
        income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
        commute = pd.to_numeric(frame["Daily_Commute_km"], errors="coerce").fillna(0)
        x["income_per_commute"] = income / (commute + 1.0)

    # Robust Yes/No -> 1/0 mapping for subsidy interactions.
    if "Subsidy_Available" in frame.columns:
        subsidy = frame["Subsidy_Available"]
        if pd.api.types.is_numeric_dtype(subsidy):
            subsidy_bin = pd.to_numeric(subsidy, errors="coerce")
        else:
            subsidy_bin = (
                subsidy.astype(str)
                .str.strip()
                .str.lower()
                .map({"yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0})
            )

        if subsidy_bin.notna().all():
            x["subsidy_bin"] = subsidy_bin.astype(np.int8).to_numpy()

            if "Annual_Income_USD" in frame.columns:
                income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
                x["subsidy_x_income"] = subsidy_bin.to_numpy() * income.to_numpy()

            if "Environmental_Concern_Level" in frame.columns:
                concern = pd.to_numeric(
                    frame["Environmental_Concern_Level"], errors="coerce"
                ).fillna(0)
                x["subsidy_x_concern"] = subsidy_bin.to_numpy() * concern.to_numpy()

            if "Charging_Stations_Near_Home" in frame.columns:
                home = pd.to_numeric(
                    frame["Charging_Stations_Near_Home"], errors="coerce"
                ).fillna(0)
                x["subsidy_x_home_charging"] = subsidy_bin.to_numpy() * home.to_numpy()

    return x


def _is_cat(series):
    return (
        pd.api.types.is_object_dtype(series)
        or pd.api.types.is_string_dtype(series)
        or isinstance(series.dtype, pd.CategoricalDtype)
    )


def encode_xgb_fold(
    train_x,
    train_y,
    valid_x,
    test_x,
    seed,
    smoothings=TE_SMOOTHINGS,
    inner_splits=INNER_TE_SPLITS,
):
    """Fold-local frequency + inner-OOF multi-scale target encoding."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y, dtype=np.int8)

    original_cols = train_x.columns.tolist()
    cat_cols = [col for col in original_cols if _is_cat(a[col])]
    numeric_exact = [col for col in NUMERIC if col in a.columns]

    # Avoid duplicate keys while keeping deterministic order.
    te_keys = list(dict.fromkeys(cat_cols + numeric_exact))

    inner = list(
        StratifiedKFold(
            inner_splits,
            shuffle=True,
            random_state=seed,
        ).split(a, labels)
    )

    for col in te_keys:
        for smoothing in smoothings:
            encoded = np.full(len(a), np.nan, dtype=np.float32)

            for fit_idx, hold_idx in inner:
                values = a[col].iloc[fit_idx].to_numpy()
                yy = labels[fit_idx]
                prior = float(yy.mean())

                stats = (
                    pd.DataFrame({"key": values, "y": yy})
                    .groupby("key", dropna=False)["y"]
                    .agg(["sum", "count"])
                )
                mapping = (
                    stats["sum"] + smoothing * prior
                ) / (
                    stats["count"] + smoothing
                )

                encoded[hold_idx] = (
                    a[col].iloc[hold_idx]
                    .map(mapping)
                    .fillna(prior)
                    .to_numpy(dtype=np.float32)
                )

            prior = float(labels.mean())
            stats = (
                pd.DataFrame({"key": a[col].to_numpy(), "y": labels})
                .groupby("key", dropna=False)["y"]
                .agg(["sum", "count"])
            )
            mapping = (
                stats["sum"] + smoothing * prior
            ) / (
                stats["count"] + smoothing
            )

            tag = str(smoothing).replace(".", "p")
            a[f"{col}_te_{tag}"] = encoded
            b[f"{col}_te_{tag}"] = (
                b[col].map(mapping).fillna(prior).to_numpy(dtype=np.float32)
            )
            c[f"{col}_te_{tag}"] = (
                c[col].map(mapping).fillna(prior).to_numpy(dtype=np.float32)
            )

            require(np.isfinite(encoded).all(), f"{col} TE contains non-finite values")

    # Label-free frequency encoding from this outer training fold only.
    n = len(a)
    for col in te_keys:
        counts = a[col].value_counts(dropna=False)
        a[f"{col}_freq"] = (
            (a[col].map(counts) - 1).clip(lower=0) / max(n - 1, 1)
        ).astype(np.float32)
        b[f"{col}_freq"] = (
            b[col].map(counts).fillna(0) / n
        ).astype(np.float32)
        c[f"{col}_freq"] = (
            c[col].map(counts).fillna(0) / n
        ).astype(np.float32)

    # XGBoost needs numeric input here. Encode raw categoricals using
    # categories learned ONLY from the outer training fold.
    for col in cat_cols:
        train_values = a[col].astype(str)
        categories = pd.Index(pd.unique(train_values))
        mapping = {value: idx for idx, value in enumerate(categories)}

        for frame in (a, b, c):
            frame[col] = (
                frame[col].astype(str)
                .map(mapping)
                .fillna(-1)
                .astype(np.int32)
            )

    for frame in (a, b, c):
        for col in frame.columns:
            if pd.api.types.is_bool_dtype(frame[col]):
                frame[col] = frame[col].astype(np.int8)

        bad = frame.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        require(not bad, f"Non-numeric XGB columns remain: {bad}")

        require(np.isfinite(frame.to_numpy(dtype=np.float64, copy=False)).all(),
                "Non-finite values in XGB feature matrix")

    require(a.columns.tolist() == b.columns.tolist() == c.columns.tolist(),
            "XGB train/valid/test feature columns differ")

    return a, b, c


## 8. 训练 10-fold XGBoost

这一步是 V5 的主要训练。

预期现象不是“每一折都一定超过你原来的 5-fold 模型”，而是：

- overall XGB OOF 尽量接近或超过 `0.9460`；
- 更理想是进入 `0.9462+`；
- 如果能靠近 heuljax 公布的 `0.9465` CV 区域，再考虑小幅调参；
- 如果 XGB OOF 本身一般，但与 Cat/LGB 的 OOF rank correlation 下降，仍可能通过 blend 提升。

训练完会保存每折模型、OOF、test prediction 和配置。


In [9]:
def run_xgb10(project, result_dir=None):
    try:
        import xgboost as xgb
    except ImportError as error:
        raise RuntimeError(
            "当前内核缺少 xgboost。请在单独一格运行 %pip install xgboost，"
            "安装后重启 kernel。"
        ) from error

    train, test, y, _ = load_project(project)
    hashes_before = input_hashes(project)

    x_all = add_xgb_row_features(train)
    x_test = add_xgb_row_features(test)
    require(x_all.columns.tolist() == x_test.columns.tolist(),
            "Row-local XGB train/test features differ")

    folds10 = list(
        StratifiedKFold(
            XGB_SPLITS,
            shuffle=True,
            random_state=XGB_SEED,
        ).split(train, y)
    )

    if result_dir is not None:
        out = Path(result_dir)
        config_path = out / "config.json"
        if not config_path.is_file():
            raise FileNotFoundError(f"Missing {config_path}")

        config = json.loads(config_path.read_text())
        require(config.get("folds") == XGB_SPLITS, "Saved XGB fold count differs")
        require(config.get("seed") == XGB_SEED, "Saved XGB seed differs")
        require(config.get("input_sha256") == hashes_before, "Saved XGB data hashes differ")

        oof_frame = align_csv(out / "oof_xgboost.csv", train.id, ["target", "prediction"])
        test_frame = align_csv(out / "submission_xgboost.csv", test.id, [TARGET])

        require(np.array_equal(oof_frame["target"].to_numpy(), y.to_numpy()),
                "Saved XGB OOF labels differ from current train.csv")

        oof = probabilities(oof_frame["prediction"], len(train), "XGB OOF")
        test_pred = probabilities(test_frame[TARGET], len(test), "XGB test")
        score = float(roc_auc_score(y, oof))

        print("复用 XGBoost：", out)
        print("XGB OOF AUC:", score)
        return {
            "dir": out,
            "oof": oof,
            "test": test_pred,
            "auc": score,
            "folds": folds10,
        }

    out = (
        Path(project)
        / "outputs"
        / ("v5_xgb10_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    )
    out.mkdir(parents=True, exist_ok=False)

    config = {
        "model": "xgboost",
        "folds": XGB_SPLITS,
        "seed": XGB_SEED,
        "inner_te_splits": INNER_TE_SPLITS,
        "te_smoothings": list(TE_SMOOTHINGS),
        "xgb_params": XGB_PARAMS,
        "train_rows": len(train),
        "test_rows": len(test),
        "input_sha256": hashes_before,
        "note": (
            "10-fold XGB inspired by public S6E9 discussion. "
            "All target encoding is nested inside outer training folds."
        ),
    }
    (out / "config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    oof = np.full(len(train), np.nan, dtype=np.float64)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test), dtype=np.float64)
    fold_rows = []

    for fold, (ti, vi) in enumerate(folds10, start=1):
        print(f"\\n========== XGBoost | Fold {fold}/{XGB_SPLITS} ==========", flush=True)

        a0 = x_all.iloc[ti]
        b0 = x_all.iloc[vi]
        c0 = x_test

        a, b, c = encode_xgb_fold(
            a0,
            y.iloc[ti],
            b0,
            c0,
            seed=3000 + fold,
        )

        params = dict(XGB_PARAMS)
        params["random_state"] = 1000 + fold

        model = xgb.XGBClassifier(**params)
        model.fit(
            a,
            y.iloc[ti],
            eval_set=[(b, y.iloc[vi])],
            verbose=200,
        )

        vp = model.predict_proba(b)[:, 1]
        tp = model.predict_proba(c)[:, 1]

        require(np.isfinite(vp).all() and np.isfinite(tp).all(),
                f"Fold {fold}: non-finite predictions")

        oof[vi] = vp
        coverage[vi] += 1
        test_sum += tp

        fold_auc = float(roc_auc_score(y.iloc[vi], vp))
        best_iteration = getattr(model, "best_iteration", None)

        fold_rows.append({
            "fold": fold,
            "auc": fold_auc,
            "best_iteration": None if best_iteration is None else int(best_iteration),
            "n_features": int(a.shape[1]),
        })

        model.save_model(str(out / f"xgboost_fold{fold}.json"))

        np.savez_compressed(
            out / f"xgboost_fold{fold}_predictions.npz",
            valid_indices=vi,
            valid_ids=train.id.iloc[vi].to_numpy(),
            valid_prediction=vp,
            test_ids=test.id.to_numpy(),
            test_prediction=tp,
        )

        pd.DataFrame(fold_rows).to_csv(out / "fold_scores.csv", index=False)

        print(f"Fold {fold} AUC: {fold_auc:.9f}")
        print("Best iteration:", best_iteration)

        del a0, b0, c0, a, b, c, model, vp, tp
        gc.collect()

    require((coverage == 1).all(), "XGB OOF coverage is not exactly one")
    require(np.isfinite(oof).all(), "XGB OOF has missing values")
    require(input_hashes(project) == hashes_before, "Input files changed during XGB training")

    test_pred = test_sum / XGB_SPLITS
    total_auc = float(roc_auc_score(y, oof))

    pd.DataFrame({
        "id": train.id.to_numpy(),
        "target": y.to_numpy(),
        "prediction": oof,
    }).to_csv(out / "oof_xgboost.csv", index=False)

    submission = pd.DataFrame({
        "id": test.id.to_numpy(),
        TARGET: test_pred,
    })
    sample_path = Path(project) / "data/sample_submission.csv"
    if sample_path.exists():
        sample_ids = pd.read_csv(sample_path, usecols=["id"])
        submission = sample_ids.merge(
            submission,
            on="id",
            how="left",
            validate="one_to_one",
            sort=False,
        )

    probabilities(submission[TARGET], len(test), "XGB submission")
    submission.to_csv(out / "submission_xgboost.csv", index=False)

    (out / "summary.json").write_text(
        json.dumps({
            "oof_auc": total_auc,
            "fold_scores": fold_rows,
            "reported_public_before_v5": CURRENT_PUBLIC_SCORE,
        }, indent=2),
        encoding="utf-8",
    )

    print("\\nXGBoost 10-fold OOF AUC:", total_auc)
    print("Result directory:", out)
    print("Single-model submission:", out / "submission_xgboost.csv")

    return {
        "dir": out,
        "oof": oof,
        "test": test_pred,
        "auc": total_auc,
        "folds": folds10,
    }


if XGB_RESULT_DIR is None:
    xgb_result = run_xgb10(PROJECT)
    XGB_RESULT_DIR = xgb_result["dir"]
else:
    xgb_result = run_xgb10(PROJECT, XGB_RESULT_DIR)


复用 XGBoost： /Users/yw/unsw/Kaggle/outputs/v5_xgb10_20260930_205349_440980
XGB OOF AUC: 0.9459315426788591


## 9. 和你当前 0.94618 路线做概率融合 + Rank Blend

AUC 只关心排序，因此当不同模型的概率尺度不一致时，rank blend 往往比直接平均概率更稳定。

这里**不做上百个权重的搜索**，只预先比较少量候选：

- 你当前 CatBoost/LGB 50/50；
- XGBoost 单模型；
- 70% XGB + 30% 当前模型（probability）；
- 70% XGB + 30% 当前模型（rank）；
- 50% XGB + 50% 当前模型（rank）。

同时输出 XGB 与当前模型的 Spearman rank correlation。相关性越低，XGB 越可能提供真正的新信息。


In [10]:
def rank01(values):
    values = np.asarray(values, dtype=np.float64)
    return pd.Series(values).rank(method="average", pct=True).to_numpy(dtype=np.float64)


def compare_v5_xgb_blends(
    project,
    train,
    test,
    y,
    old,
    new,
    xgb_result,
):
    # Current best development route from the uploaded notebook.
    current_oof = 0.5 * old["oof"] + 0.5 * new["oof"]
    current_test = 0.5 * old["test"] + 0.5 * new["test"]

    xgb_oof = xgb_result["oof"]
    xgb_test = xgb_result["test"]

    current_rank_oof = rank01(current_oof)
    xgb_rank_oof = rank01(xgb_oof)
    current_rank_test = rank01(current_test)
    xgb_rank_test = rank01(xgb_test)

    spearman_corr = float(
        pd.Series(current_oof).corr(pd.Series(xgb_oof), method="spearman")
    )
    pearson_corr = float(np.corrcoef(current_oof, xgb_oof)[0, 1])

    candidates = {
        "current_cat50_lgb50": (
            current_oof,
            current_test,
        ),
        "xgb_single": (
            xgb_oof,
            xgb_test,
        ),
        "xgb70_current30_prob": (
            0.70 * xgb_oof + 0.30 * current_oof,
            0.70 * xgb_test + 0.30 * current_test,
        ),
        "xgb70_current30_rank": (
            0.70 * xgb_rank_oof + 0.30 * current_rank_oof,
            0.70 * xgb_rank_test + 0.30 * current_rank_test,
        ),
        "xgb50_current50_rank": (
            0.50 * xgb_rank_oof + 0.50 * current_rank_oof,
            0.50 * xgb_rank_test + 0.50 * current_rank_test,
        ),
    }

    out = (
        Path(project)
        / "outputs"
        / ("v5_compare_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    )
    out.mkdir(parents=True, exist_ok=False)

    rows = []
    per_fold = []
    folds10 = xgb_result["folds"]

    current_auc = float(roc_auc_score(y, current_oof))
    xgb_auc = float(roc_auc_score(y, xgb_oof))

    for name, (oof_pred, test_pred) in candidates.items():
        auc = float(roc_auc_score(y, oof_pred))

        fold_deltas_vs_current = []
        for fold, (_, vi) in enumerate(folds10, start=1):
            candidate_fold_auc = float(roc_auc_score(y.iloc[vi], oof_pred[vi]))
            current_fold_auc = float(roc_auc_score(y.iloc[vi], current_oof[vi]))
            delta = candidate_fold_auc - current_fold_auc

            fold_deltas_vs_current.append(delta)
            per_fold.append({
                "candidate": name,
                "fold": fold,
                "auc": candidate_fold_auc,
                "delta_vs_current": delta,
            })

        rows.append({
            "candidate": name,
            "oof_auc": auc,
            "delta_vs_current": auc - current_auc,
            "delta_vs_xgb": auc - xgb_auc,
            "improved_xgb_folds_vs_current": int(
                sum(d > 0 for d in fold_deltas_vs_current)
            ),
            "worst_fold_delta_vs_current": float(min(fold_deltas_vs_current)),
        })

        pd.DataFrame({
            "id": train.id.to_numpy(),
            "target": y.to_numpy(),
            "prediction": oof_pred,
        }).to_csv(out / f"oof_{name}.csv", index=False)

        sub = pd.DataFrame({
            "id": test.id.to_numpy(),
            TARGET: probabilities(test_pred, len(test), name),
        })

        sample_path = Path(project) / "data/sample_submission.csv"
        if sample_path.exists():
            sample_ids = pd.read_csv(sample_path, usecols=["id"])
            sub = sample_ids.merge(
                sub,
                on="id",
                how="left",
                validate="one_to_one",
                sort=False,
            )

        probabilities(sub[TARGET], len(test), f"{name} submission")
        sub.to_csv(out / f"submission_{name}.csv", index=False)

    table = (
        pd.DataFrame(rows)
        .sort_values("oof_auc", ascending=False, kind="stable")
        .reset_index(drop=True)
    )
    table.to_csv(out / "comparison.csv", index=False)
    pd.DataFrame(per_fold).to_csv(out / "comparison_by_fold.csv", index=False)

    best_name = table.iloc[0]["candidate"]
    best_submission = pd.read_csv(out / f"submission_{best_name}.csv")
    best_submission.to_csv(out / "submission_next_candidate.csv", index=False)

    diagnostics = {
        "current_oof_auc": current_auc,
        "xgb_oof_auc": xgb_auc,
        "reported_current_public": CURRENT_PUBLIC_SCORE,
        "spearman_xgb_vs_current": spearman_corr,
        "pearson_xgb_vs_current": pearson_corr,
        "selected_by_oof": best_name,
    }
    (out / "diagnostics.json").write_text(
        json.dumps(diagnostics, indent=2),
        encoding="utf-8",
    )

    print("Current Cat/LGB OOF:", current_auc)
    print("XGB OOF:", xgb_auc)
    print("Spearman(XGB, current):", spearman_corr)
    print("Pearson(XGB, current):", pearson_corr)
    print()
    print(table.to_string(index=False, float_format=lambda v: f"{v:.9f}"))
    print("\\nComparison directory:", out)
    print("OOF-selected candidate:", best_name)
    print("Candidate submission:", out / "submission_next_candidate.csv")
    print("\\n也保留了 XGB 单模型文件：", xgb_result["dir"] / "submission_xgboost.csv")

    return out, table, diagnostics


# Re-load/verify the saved 5-fold models from your existing V4 state.
train, test, y, folds = load_project(PROJECT)
old = read_run(
    BASELINE_DIR,
    "catboost",
    train,
    test,
    y,
    folds,
    expected_auc=EXPECTED_BASELINE_AUC,
)
new = read_run(
    Path(LGB_RESULT_DIR),
    "lightgbm",
    train,
    test,
    y,
    folds,
)

v5_compare_dir, v5_comparison, v5_diagnostics = compare_v5_xgb_blends(
    PROJECT,
    train,
    test,
    y,
    old,
    new,
    xgb_result,
)

v5_comparison


Current Cat/LGB OOF: 0.945986243390633
XGB OOF: 0.9459315426788591
Spearman(XGB, current): 0.9963674136703535
Pearson(XGB, current): 0.9979214260945515

           candidate     oof_auc  delta_vs_current  delta_vs_xgb  improved_xgb_folds_vs_current  worst_fold_delta_vs_current
xgb50_current50_rank 0.946098745       0.000112502   0.000167203                             10                  0.000061974
xgb70_current30_rank 0.946064667       0.000078424   0.000133124                             10                  0.000014073
xgb70_current30_prob 0.946060821       0.000074577   0.000129278                             10                  0.000002141
 current_cat50_lgb50 0.945986243       0.000000000   0.000054701                              0                  0.000000000
          xgb_single 0.945931543      -0.000054701   0.000000000                              2                 -0.000139096
\nComparison directory: /Users/yw/unsw/Kaggle/outputs/v5_compare_20260930_213729_747163
OOF-selec

,candidate,oof_auc,delta_vs_current,delta_vs_xgb,improved_xgb_folds_vs_current,worst_fold_delta_vs_current
0,xgb50_current50_rank,0.946099,0.000113,0.000167,10,0.000062
1,xgb70_current30_rank,0.946065,0.000078,0.000133,10,0.000014
2,xgb70_current30_prob,0.946061,0.000075,0.000129,10,0.000002
3,current_cat50_lgb50,0.945986,0.000000,0.000055,0,0.000000
4,xgb_single,0.945932,-0.000055,0.000000,2,-0.000139


## 11. V7 设置

这次 meta ensemble 用新的 **5-fold meta CV**。

注意这里的输入已经是基础模型的 **OOF predictions**，所以 meta fold 只负责：

- 在 meta-training rows 上选择 hill-climb 权重；
- 在 meta-validation rows 上评估这些权重；
- 训练 Logistic stacker 并在 meta-validation rows 上预测。

因此 hill climb / stacker 不会直接在它自己评估的标签上拟合。

`META_MIN_ENGINE_AUC` 只是一个很宽松的门槛；真正的最终 gate 是：

- 至少 **4/5 folds** 比 V5 baseline 好；
- 平均 fold delta > 0。


In [11]:
# ===== V7 meta-ensemble configuration =====
META_FOLDS = 5
META_SEED = 2026

HILL_MAX_STEPS = 80
HILL_TOL = 1e-7
META_MIN_ENGINE_AUC = 0.9450

STACK_C = 0.01
TEAM_HILL_WEIGHT = 0.75
TEAM_STACK_WEIGHT = 0.25

# 外部高分 notebook 使用的 4 条 generator rules。
# V7 会单独报告纯 OOF 版本和 rules 版本。
ENABLE_PUBLISHED_RULES = True

V5_REFERENCE_OOF = 0.9460987453734608

print("V7 meta folds:", META_FOLDS)
print("Hill max steps:", HILL_MAX_STEPS)
print("Stack C:", STACK_C)
print("Published rules enabled:", ENABLE_PUBLISHED_RULES)


V7 meta folds: 5
Hill max steps: 80
Stack C: 0.01
Published rules enabled: True


## 12. 构建自己的 “engine garage”

高分 notebook 有 83 个 engine；我们现在没有那么多独立模型，所以先从已有预测构造一个**小而透明**的 garage。

基础 engine：

- CatBoost
- LightGBM
- XGBoost

再加入几个已经验证/合理的无标签 blend view：

- Cat/LGB probability mean
- Cat/XGB probability mean
- LGB/XGB probability mean
- 三模型 probability mean
- V5：XGB + Current 的 50/50 rank blend
- 三模型 equal-rank blend

这些 blend 本身不增加新的原始信息，但它们可以形成不同的**排序边界**。Nested meta CV 会决定它们是否真的有用；没用的不会通过 gate。


In [12]:
from scipy.stats import rankdata, norm
from sklearn.linear_model import LogisticRegression


def v7_rank01(values):
    values = np.asarray(values, dtype=np.float64)
    require(np.isfinite(values).all(), "rank01 input contains non-finite values")
    return rankdata(values, method="average") / len(values)


def v7_probit(values):
    """
    Convert model scores to rank-normalized Gaussian scale.
    Monotonic -> single-model AUC is unchanged.
    """
    values = np.asarray(values, dtype=np.float64)
    require(np.isfinite(values).all(), "probit input contains non-finite values")
    r = rankdata(values, method="average")
    return norm.ppf(r / (len(values) + 1.0)).astype(np.float64)


def v7_zscore(values):
    values = np.asarray(values, dtype=np.float64)
    sd = float(values.std())
    require(sd > 0, "Cannot z-score a constant vector")
    return (values - values.mean()) / sd


def build_v7_engines(old, new, xgb_result):
    cat_oof, cat_test = old["oof"], old["test"]
    lgb_oof, lgb_test = new["oof"], new["test"]
    xgb_oof, xgb_test = xgb_result["oof"], xgb_result["test"]

    current_oof = 0.50 * cat_oof + 0.50 * lgb_oof
    current_test = 0.50 * cat_test + 0.50 * lgb_test

    v5_oof = (
        0.50 * v7_rank01(xgb_oof)
        + 0.50 * v7_rank01(current_oof)
    )
    v5_test = (
        0.50 * v7_rank01(xgb_test)
        + 0.50 * v7_rank01(current_test)
    )

    equal_rank_oof = (
        v7_rank01(cat_oof)
        + v7_rank01(lgb_oof)
        + v7_rank01(xgb_oof)
    ) / 3.0
    equal_rank_test = (
        v7_rank01(cat_test)
        + v7_rank01(lgb_test)
        + v7_rank01(xgb_test)
    ) / 3.0

    engines_oof = {
        "catboost": cat_oof,
        "lightgbm": lgb_oof,
        "xgboost": xgb_oof,
        "cat_lgb_prob50": 0.50 * cat_oof + 0.50 * lgb_oof,
        "cat_xgb_prob50": 0.50 * cat_oof + 0.50 * xgb_oof,
        "lgb_xgb_prob50": 0.50 * lgb_oof + 0.50 * xgb_oof,
        "cat_lgb_xgb_prob_equal": (cat_oof + lgb_oof + xgb_oof) / 3.0,
        "v5_xgb50_current50_rank": v5_oof,
        "cat_lgb_xgb_equal_rank": equal_rank_oof,
    }

    engines_test = {
        "catboost": cat_test,
        "lightgbm": lgb_test,
        "xgboost": xgb_test,
        "cat_lgb_prob50": 0.50 * cat_test + 0.50 * lgb_test,
        "cat_xgb_prob50": 0.50 * cat_test + 0.50 * xgb_test,
        "lgb_xgb_prob50": 0.50 * lgb_test + 0.50 * xgb_test,
        "cat_lgb_xgb_prob_equal": (cat_test + lgb_test + xgb_test) / 3.0,
        "v5_xgb50_current50_rank": v5_test,
        "cat_lgb_xgb_equal_rank": equal_rank_test,
    }

    require(
        list(engines_oof) == list(engines_test),
        "OOF/test engine names differ",
    )

    names = list(engines_oof)

    # Probit-rank every engine separately on OOF and test, as in the reference notebook.
    p_oof = np.column_stack([v7_probit(engines_oof[n]) for n in names])
    p_test = np.column_stack([v7_probit(engines_test[n]) for n in names])

    return {
        "names": names,
        "raw_oof": engines_oof,
        "raw_test": engines_test,
        "P_OOF": p_oof,
        "P_TEST": p_test,
        "v5_oof": v5_oof,
        "v5_test": v5_test,
    }


## 13. 相关性指纹 + 单 engine OOF

Probit 不会改变每个 engine 自己的 AUC，因为它只是单调变换。

这里先打印：

- 每个 engine 的 OOF AUC；
- OOF 与 test 相对 V5 engine 的相关性差异。

如果某个 engine 在 OOF 与 test 上的相关结构差得离谱，说明可能存在文件错位或 train/test behavior mismatch。


In [25]:
def v7_engine_diagnostics(garage, y):
    names = garage["names"]
    p_oof = garage["P_OOF"]
    p_test = garage["P_TEST"]

    rows = []
    for i, name in enumerate(names):
        rows.append({
            "engine": name,
            "oof_auc": float(roc_auc_score(y, p_oof[:, i])),
        })

    single = (
        pd.DataFrame(rows)
        .sort_values("oof_auc", ascending=False, kind="stable")
        .reset_index(drop=True)
    )

    # V7 uses "v5_xgb50_current50_rank";
    # V8 renamed the same reference engine to "v5_baseline".
    if "v5_xgb50_current50_rank" in names:
        ref_name = "v5_xgb50_current50_rank"
    elif "v5_baseline" in names:
        ref_name = "v5_baseline"
    else:
        raise ValueError(
            f"Cannot find V5 reference engine. Available engines: {names}"
        )

    ref_idx = names.index(ref_name)

    fp_rows = []
    for i, name in enumerate(names):
        oof_corr = float(np.corrcoef(p_oof[:, i], p_oof[:, ref_idx])[0, 1])
        test_corr = float(np.corrcoef(p_test[:, i], p_test[:, ref_idx])[0, 1])
        fp_rows.append({
            "engine": name,
            "oof_corr_vs_v5": oof_corr,
            "test_corr_vs_v5": test_corr,
            "test_minus_oof": test_corr - oof_corr,
        })

    fingerprint = (
        pd.DataFrame(fp_rows)
        .sort_values("test_minus_oof")
        .reset_index(drop=True)
    )

    print("Single-engine OOF AUC:")
    print(single.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

    print("\nCorrelation fingerprint vs V5:")
    print(fingerprint.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

    return single, fingerprint


## 14. Nested Caruana Hill Climb

这部分基本复现高分 notebook 的核心 ensemble selection：

每一步都尝试：

`当前累计 blend + 某一个 engine`

选择训练 meta-fold 上 AUC 最大的 engine 加进去；同一个 engine 可以重复出现，因此最终“被选次数 / 总次数”就是权重。

关键区别是：**权重只在 meta-training rows 上选择，meta-validation rows 只用于评估。**

我们把 V5 作为真正的 reference：

只有 nested hill 在至少 4/5 folds 超过 V5，且平均 delta > 0，才认为 hill 通过 gate。


In [26]:
def v7_hill_climb(P, labels, candidates, max_steps=HILL_MAX_STEPS, tol=HILL_TOL):
    P = np.asarray(P, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)

    blend = np.zeros(len(P), dtype=np.float64)
    counts = np.zeros(P.shape[1], dtype=np.float64)
    best = -np.inf

    for _ in range(max_steps):
        trial_scores = {}
        for j in candidates:
            trial_scores[j] = float(roc_auc_score(labels, blend + P[:, j]))

        j = max(trial_scores, key=trial_scores.get)
        score = trial_scores[j]

        if score <= best + tol:
            break

        best = score
        blend += P[:, j]
        counts[j] += 1.0

    require(counts.sum() > 0, "Hill climb selected no engine")
    return counts / counts.sum(), float(best)


def run_v7_nested_hill(garage, y):
    names = garage["names"]
    P = garage["P_OOF"]
    v5_oof = garage["v5_oof"]

    single_auc = np.array([
        roc_auc_score(y, P[:, j])
        for j in range(P.shape[1])
    ])

    candidates = [
        j for j, auc in enumerate(single_auc)
        if auc >= META_MIN_ENGINE_AUC
    ]
    require(candidates, "No V7 hill-climb candidates pass the AUC threshold")

    folds = list(
        StratifiedKFold(
            META_FOLDS,
            shuffle=True,
            random_state=META_SEED,
        ).split(P, y)
    )

    hill_oof = np.zeros(len(y), dtype=np.float64)
    fold_rows = []
    fold_weights = []

    for fold, (tr, va) in enumerate(folds, start=1):
        w, train_auc = v7_hill_climb(
            P[tr],
            np.asarray(y)[tr],
            candidates,
        )

        pred = P[va] @ w
        hill_oof[va] = pred

        hill_auc = float(roc_auc_score(np.asarray(y)[va], pred))
        v5_auc = float(roc_auc_score(np.asarray(y)[va], v5_oof[va]))
        delta = hill_auc - v5_auc

        fold_rows.append({
            "fold": fold,
            "train_hill_auc_optimistic": train_auc,
            "v5_auc": v5_auc,
            "nested_hill_auc": hill_auc,
            "delta_vs_v5": delta,
            "engines_used": int((w > 0).sum()),
        })

        fold_weights.append(pd.Series(w, index=names, name=f"fold_{fold}"))

        print(
            f"Fold {fold}: hill {hill_auc:.9f} | "
            f"V5 {v5_auc:.9f} | delta {delta:+.9f} | "
            f"{int((w > 0).sum())} engines"
        )

    fold_table = pd.DataFrame(fold_rows)
    weight_table = pd.concat(fold_weights, axis=1).fillna(0.0)

    wins = int((fold_table["delta_vs_v5"] > 0).sum())
    mean_delta = float(fold_table["delta_vs_v5"].mean())
    accepted = bool(wins >= 4 and mean_delta > 0)

    nested_auc = float(roc_auc_score(y, hill_oof))
    v5_auc = float(roc_auc_score(y, v5_oof))

    # Full-data hill is used ONLY to create test weights after nested validation.
    w_full, full_train_auc = v7_hill_climb(
        P,
        np.asarray(y),
        candidates,
    )

    full_weights = (
        pd.Series(w_full, index=names, name="weight")
        .loc[lambda s: s > 0]
        .sort_values(ascending=False)
    )

    print("\nNested hill OOF:", nested_auc)
    print("V5 OOF:", v5_auc)
    print("Delta:", nested_auc - v5_auc)
    print(f"Wins vs V5: {wins}/{META_FOLDS}")
    print("Accepted:", accepted)

    print("\nFull-data hill weights (used for test only if nested gate passes):")
    print(full_weights.to_string(float_format=lambda v: f"{v:.4f}"))

    return {
        "folds": folds,
        "oof": hill_oof,
        "auc": nested_auc,
        "accepted": accepted,
        "wins": wins,
        "mean_delta": mean_delta,
        "fold_table": fold_table,
        "fold_weights": weight_table,
        "w_full": w_full,
        "full_train_auc_optimistic": full_train_auc,
    }


## 15. OOF Logistic Stack + gated team car

注意这里的 Logistic Regression 和上一版失败的 raw-feature LR 完全不同。

它的输入是：

`[CatBoost prediction, LightGBM prediction, XGBoost prediction, blend engines...]`

而不是 EV 的原始字段。

每个 meta fold：

- 用 meta-training OOF matrix 拟合 `LogisticRegression(C=0.01)`；
- 在 meta-validation rows 上输出 decision function；
- 拼起来得到完整 `stack_oof`。

然后测试参考 notebook 的：

`75% zscore(hill) + 25% zscore(stack)`

只有 team 在至少 4/5 folds 比 hill 更好、且平均 gain > 0 时才采用。


In [27]:
def run_v7_stack_and_team(garage, y, hill_result):
    P = garage["P_OOF"]
    folds = hill_result["folds"]

    stack_oof = np.zeros(len(y), dtype=np.float64)
    stack_rows = []

    for fold, (tr, va) in enumerate(folds, start=1):
        model = LogisticRegression(
            C=STACK_C,
            penalty="l2",
            solver="lbfgs",
            max_iter=400,
        )
        model.fit(P[tr], np.asarray(y)[tr])
        pred = model.decision_function(P[va])
        stack_oof[va] = pred

        stack_auc = float(roc_auc_score(np.asarray(y)[va], pred))
        v5_auc = float(
            roc_auc_score(
                np.asarray(y)[va],
                garage["v5_oof"][va],
            )
        )
        stack_rows.append({
            "fold": fold,
            "stack_auc": stack_auc,
            "v5_auc": v5_auc,
            "delta_vs_v5": stack_auc - v5_auc,
        })

    stack_auc = float(roc_auc_score(y, stack_oof))

    # "Hill source" for the team:
    # if nested hill did not pass the V5 gate, use V5 itself instead.
    if hill_result["accepted"]:
        hill_source_oof = hill_result["oof"]
        hill_label = "accepted_nested_hill"
    else:
        hill_source_oof = garage["v5_oof"]
        hill_label = "v5_fallback"

    team_oof = (
        TEAM_HILL_WEIGHT * v7_zscore(hill_source_oof)
        + TEAM_STACK_WEIGHT * v7_zscore(stack_oof)
    )
    team_auc = float(roc_auc_score(y, team_oof))
    hill_source_auc = float(roc_auc_score(y, hill_source_oof))

    team_fold_gains = []
    for fold, (_, va) in enumerate(folds, start=1):
        before = float(
            roc_auc_score(
                np.asarray(y)[va],
                hill_source_oof[va],
            )
        )
        after = float(
            roc_auc_score(
                np.asarray(y)[va],
                team_oof[va],
            )
        )
        team_fold_gains.append(after - before)

    team_wins = int(sum(g > 0 for g in team_fold_gains))
    team_accepted = bool(
        team_wins >= 4
        and float(np.mean(team_fold_gains)) > 0
    )

    full_stack = LogisticRegression(
        C=STACK_C,
        penalty="l2",
        solver="lbfgs",
        max_iter=400,
    ).fit(P, y)

    print("Logistic stack OOF:", stack_auc)
    print("Hill source:", hill_label, "| AUC:", hill_source_auc)
    print("Team OOF:", team_auc)
    print(
        "Team fold gains vs hill source:",
        " ".join(f"{g:+.9f}" for g in team_fold_gains)
    )
    print(f"Team wins: {team_wins}/{META_FOLDS}")
    print("Team accepted:", team_accepted)

    return {
        "stack_oof": stack_oof,
        "stack_auc": stack_auc,
        "stack_fold_table": pd.DataFrame(stack_rows),
        "full_stack_model": full_stack,
        "hill_source_oof": hill_source_oof,
        "hill_source_label": hill_label,
        "team_oof": team_oof,
        "team_auc": team_auc,
        "team_fold_gains": team_fold_gains,
        "team_wins": team_wins,
        "team_accepted": team_accepted,
    }


## 16. 四个 published generator hard rules

参考 notebook 使用的规则：

1. `income >= 170537` → positive block
2. `31004 <= income <= 41970` → negative block
3. `commute >= 83` → negative block
4. `income == 30000` 且无补贴，并且 concern=1 或 anxiety∈{Medium, High} → negative block

这里会：

- 打印每个 rule 在你 train 上的样本数和 positive rate；
- 在 5 个 meta folds 上分别比较规则前/后 AUC；
- 只有**每折都不下降**且平均提升 > 0 才标记 accepted。

⚠️ **验证解释要谨慎：**这些规则本身是社区先观察完整训练数据后公开的，因此这里的 rule OOF 不是完全独立的“规则发现验证”。所以 V7 会同时保存：

- `OOF-only` submission：不使用 hard rules；
- `rules-experimental` submission：使用公开规则。

这样你不会把两种证据混在一起。


In [28]:
def v7_rule_masks(frame):
    income = pd.to_numeric(
        frame["Annual_Income_USD"],
        errors="coerce",
    ).to_numpy(np.float64)

    commute = pd.to_numeric(
        frame["Daily_Commute_km"],
        errors="coerce",
    ).to_numpy(np.float64)

    no_subsidy = (
        frame["Subsidy_Available"]
        .astype(str)
        .str.strip()
        .str.lower()
        .to_numpy()
        == "no"
    )

    concern_1 = (
        pd.to_numeric(
            frame["Environmental_Concern_Level"],
            errors="coerce",
        ).to_numpy()
        == 1
    )

    anxious = np.isin(
        frame["Range_Anxiety_Level"].astype(str).str.strip().to_numpy(),
        ["Medium", "High"],
    )

    return [
        (
            "income >= 170537 -> always buys",
            income >= 170537,
            +1,
        ),
        (
            "income 31004-41970 -> never buys",
            (income >= 31004) & (income <= 41970),
            -1,
        ),
        (
            "commute >= 83 -> never buys",
            commute >= 83,
            -1,
        ),
        (
            "30k + no subsidy + concern1/anxious -> never buys",
            (income == 30000)
            & no_subsidy
            & (concern_1 | anxious),
            -1,
        ),
    ]


def v7_apply_rules(score, frame):
    """
    Put published positive rule cells above everything else,
    negative rule cells below everything else,
    while preserving within-cell order.
    """
    r = v7_rank01(score)

    for _, mask, sign in v7_rule_masks(frame):
        r = r.copy()
        r[mask] += 2.0 * sign

    return v7_rank01(r)


def evaluate_v7_rules(score_oof, train, y, folds):
    base_rank = v7_rank01(score_oof)

    print("Published rule purity on current train:")
    rule_stats = []

    for label, mask, sign in v7_rule_masks(train):
        n = int(mask.sum())
        positives = int(np.asarray(y)[mask].sum())
        rate = float(np.asarray(y)[mask].mean()) if n else np.nan

        rule_stats.append({
            "rule": label,
            "direction": sign,
            "rows": n,
            "positives": positives,
            "positive_rate": rate,
        })

    rule_stats = pd.DataFrame(rule_stats)
    print(
        rule_stats.to_string(
            index=False,
            float_format=lambda v: f"{v:.6f}",
        )
    )

    fold_rows = []

    for fold, (_, va) in enumerate(folds, start=1):
        before = float(
            roc_auc_score(
                np.asarray(y)[va],
                base_rank[va],
            )
        )
        after_pred = v7_apply_rules(
            score_oof[va],
            train.iloc[va],
        )
        after = float(
            roc_auc_score(
                np.asarray(y)[va],
                after_pred,
            )
        )

        fold_rows.append({
            "fold": fold,
            "before": before,
            "after_rules": after,
            "delta": after - before,
        })

    fold_table = pd.DataFrame(fold_rows)

    accepted = bool(
        ENABLE_PUBLISHED_RULES
        and (fold_table["delta"] >= 0).all()
        and float(fold_table["delta"].mean()) > 0
    )

    ruled_oof = (
        v7_apply_rules(score_oof, train)
        if accepted
        else base_rank
    )

    print("\nRule fold deltas:")
    print(
        " ".join(
            f"{v:+.9f}"
            for v in fold_table["delta"].to_numpy()
        )
    )
    print("Mean:", float(fold_table["delta"].mean()))
    print("Rules accepted:", accepted)
    print(
        "AUC before:",
        float(roc_auc_score(y, score_oof)),
        "| after/fallback:",
        float(roc_auc_score(y, ruled_oof)),
    )

    return {
        "rule_stats": rule_stats,
        "fold_table": fold_table,
        "accepted": accepted,
        "oof": ruled_oof,
        "auc": float(roc_auc_score(y, ruled_oof)),
    }


## 17. 一次运行 V7，并导出所有候选 submission

输出目录里会有：

- `submission_v7_v5_baseline.csv`
- `submission_v7_nested_hill.csv`
- `submission_v7_logistic_stack.csv`
- `submission_v7_oof_only_team.csv`
- `submission_v7_rules_experimental.csv`
- `submission_v7_next_candidate.csv`

其中：

### `submission_v7_next_candidate.csv`

只根据**meta OOF gate**在 V5 / hill / team 中选择，不使用 Public LB。

### `submission_v7_rules_experimental.csv`

在 OOF-only 最终候选上再加 published hard rules。它单独保存，不会悄悄覆盖纯 OOF 候选。


In [29]:
def save_v7_submission(project, test, values, path):
    values = np.asarray(values, dtype=np.float64)
    require(len(values) == len(test), f"{path.name}: wrong test length")
    require(np.isfinite(values).all(), f"{path.name}: non-finite predictions")

    # Rank to [0,1] so all meta scores become valid probabilities/order scores.
    values = v7_rank01(values)

    sub = pd.DataFrame({
        "id": test["id"].to_numpy(),
        TARGET: values,
    })

    sample_path = Path(project) / "data/sample_submission.csv"
    if sample_path.exists():
        sample_ids = pd.read_csv(sample_path, usecols=["id"])
        sub = sample_ids.merge(
            sub,
            on="id",
            how="left",
            validate="one_to_one",
            sort=False,
        )

    require(sub[TARGET].notna().all(), f"{path.name}: missing predictions after ID alignment")
    require(sub[TARGET].between(0, 1).all(), f"{path.name}: predictions outside [0,1]")
    sub.to_csv(path, index=False)


def run_v7(project):
    train, test, y, folds5 = load_project(project)

    old = read_run(
        BASELINE_DIR,
        "catboost",
        train,
        test,
        y,
        folds5,
        expected_auc=EXPECTED_BASELINE_AUC,
    )

    new = read_run(
        Path(LGB_RESULT_DIR),
        "lightgbm",
        train,
        test,
        y,
        folds5,
    )

    # Reuse the completed 10-fold XGB saved in V5.
    xgb_result = run_xgb10(
        project,
        XGB_RESULT_DIR,
    )

    garage = build_v7_engines(
        old,
        new,
        xgb_result,
    )

    print("\n================ ENGINE DIAGNOSTICS ================")
    single_table, fingerprint = v7_engine_diagnostics(
        garage,
        y,
    )

    print("\n================ NESTED HILL ================")
    hill = run_v7_nested_hill(
        garage,
        y,
    )

    print("\n================ STACK + TEAM ================")
    stack = run_v7_stack_and_team(
        garage,
        y,
        hill,
    )

    # ----- Build full-data test variants -----
    P_TEST = garage["P_TEST"]

    # Full-data hill test.
    hill_test = P_TEST @ hill["w_full"]

    # Full-data logistic stack test.
    stack_test = stack["full_stack_model"].decision_function(P_TEST)

    # If nested hill failed its gate, team starts from V5 test instead.
    if hill["accepted"]:
        hill_source_test = hill_test
    else:
        hill_source_test = garage["v5_test"]

    team_test = (
        TEAM_HILL_WEIGHT * v7_zscore(hill_source_test)
        + TEAM_STACK_WEIGHT * v7_zscore(stack_test)
    )

    # Gate team exactly as evaluated OOF.
    if stack["team_accepted"]:
        oof_only_oof = stack["team_oof"]
        oof_only_test = team_test
        oof_only_name = "team"
    elif hill["accepted"]:
        oof_only_oof = hill["oof"]
        oof_only_test = hill_test
        oof_only_name = "nested_hill"
    else:
        oof_only_oof = garage["v5_oof"]
        oof_only_test = garage["v5_test"]
        oof_only_name = "v5_fallback"

    print("\n================ PUBLISHED RULES ================")
    rules = evaluate_v7_rules(
        oof_only_oof,
        train,
        y,
        hill["folds"],
    )

    if rules["accepted"]:
        rules_test = v7_apply_rules(
            oof_only_test,
            test,
        )
    else:
        rules_test = v7_rank01(oof_only_test)

    # ----- Comparison table -----
    candidates = {
        "v5_baseline": garage["v5_oof"],
        "nested_hill": hill["oof"],
        "logistic_stack": stack["stack_oof"],
        "team_75hill_25stack": stack["team_oof"],
        "oof_gated_final": oof_only_oof,
        "rules_experimental": rules["oof"],
    }

    comparison_rows = []
    for name, pred in candidates.items():
        comparison_rows.append({
            "candidate": name,
            "oof_auc": float(roc_auc_score(y, pred)),
            "delta_vs_v5": float(
                roc_auc_score(y, pred)
                - roc_auc_score(y, garage["v5_oof"])
            ),
        })

    comparison = (
        pd.DataFrame(comparison_rows)
        .sort_values("oof_auc", ascending=False, kind="stable")
        .reset_index(drop=True)
    )

    out = (
        Path(project)
        / "outputs"
        / ("v7_meta_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    )
    out.mkdir(parents=True, exist_ok=False)

    # Save reports.
    single_table.to_csv(out / "single_engine_auc.csv", index=False)
    fingerprint.to_csv(out / "correlation_fingerprint.csv", index=False)
    hill["fold_table"].to_csv(out / "nested_hill_folds.csv", index=False)
    hill["fold_weights"].to_csv(out / "nested_hill_fold_weights.csv")
    stack["stack_fold_table"].to_csv(out / "logistic_stack_folds.csv", index=False)
    rules["rule_stats"].to_csv(out / "published_rule_stats.csv", index=False)
    rules["fold_table"].to_csv(out / "published_rule_folds.csv", index=False)
    comparison.to_csv(out / "comparison.csv", index=False)

    # Save submissions.
    save_v7_submission(
        project,
        test,
        garage["v5_test"],
        out / "submission_v7_v5_baseline.csv",
    )
    save_v7_submission(
        project,
        test,
        hill_test,
        out / "submission_v7_nested_hill.csv",
    )
    save_v7_submission(
        project,
        test,
        stack_test,
        out / "submission_v7_logistic_stack.csv",
    )
    save_v7_submission(
        project,
        test,
        team_test,
        out / "submission_v7_team_raw.csv",
    )
    save_v7_submission(
        project,
        test,
        oof_only_test,
        out / "submission_v7_oof_only_team.csv",
    )
    save_v7_submission(
        project,
        test,
        rules_test,
        out / "submission_v7_rules_experimental.csv",
    )
    save_v7_submission(
        project,
        test,
        oof_only_test,
        out / "submission_v7_next_candidate.csv",
    )

    report = {
        "v5_auc": float(roc_auc_score(y, garage["v5_oof"])),
        "nested_hill_auc": hill["auc"],
        "nested_hill_accepted": hill["accepted"],
        "nested_hill_wins_vs_v5": hill["wins"],
        "stack_auc": stack["stack_auc"],
        "team_auc": stack["team_auc"],
        "team_accepted": stack["team_accepted"],
        "team_wins": stack["team_wins"],
        "oof_only_selected": oof_only_name,
        "oof_only_auc": float(roc_auc_score(y, oof_only_oof)),
        "rules_accepted": rules["accepted"],
        "rules_auc": rules["auc"],
        "hill_full_weights": {
            garage["names"][i]: float(w)
            for i, w in enumerate(hill["w_full"])
            if w > 0
        },
    }

    (out / "v7_report.json").write_text(
        json.dumps(report, indent=2),
        encoding="utf-8",
    )

    print("\n================ V7 FINAL COMPARISON ================")
    print(
        comparison.to_string(
            index=False,
            float_format=lambda v: f"{v:.9f}",
        )
    )

    print("\nNested hill accepted:", hill["accepted"])
    print("Team accepted:", stack["team_accepted"])
    print("OOF-only selected:", oof_only_name)
    print("Published rules accepted:", rules["accepted"])

    print("\nOutput directory:", out)
    print(
        "Pure OOF candidate:",
        out / "submission_v7_next_candidate.csv",
    )
    print(
        "Rules experimental:",
        out / "submission_v7_rules_experimental.csv",
    )

    return {
        "dir": out,
        "garage": garage,
        "hill": hill,
        "stack": stack,
        "rules": rules,
        "comparison": comparison,
        "report": report,
    }


v7_result = run_v7(PROJECT)
v7_result["comparison"]


复用 XGBoost： /Users/yw/unsw/Kaggle/outputs/v5_xgb10_20260930_205349_440980
XGB OOF AUC: 0.9459315426788591

================ ENGINE DIAGNOSTICS ================
Single-engine OOF AUC:
                 engine     oof_auc
 cat_lgb_xgb_equal_rank 0.946126929
         lgb_xgb_prob50 0.946120255
v5_xgb50_current50_rank 0.946098745
 cat_lgb_xgb_prob_equal 0.946085780
         cat_lgb_prob50 0.945986243
         cat_xgb_prob50 0.945961444
               lightgbm 0.945946381
                xgboost 0.945931543
               catboost 0.945618233

Correlation fingerprint vs V5:
                 engine  oof_corr_vs_v5  test_corr_vs_v5  test_minus_oof
v5_xgb50_current50_rank        1.000000         1.000000        0.000000
 cat_lgb_xgb_prob_equal        0.999853         0.999910        0.000057
 cat_lgb_xgb_equal_rank        0.999612         0.999724        0.000112
         lgb_xgb_prob50        0.999240         0.999453        0.000213
         cat_xgb_prob50        0.999087         0.999395    

/opt/miniconda3/envs/dl/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/dl/lib/python3.11/s

Logistic stack OOF: 0.9461147258366325
Hill source: accepted_nested_hill | AUC: 0.946140989060347
Team OOF: 0.9461377169149705
Team fold gains vs hill source: -0.000002533 -0.000004795 -0.000004499 -0.000002070 -0.000001936
Team wins: 0/5
Team accepted: False

================ PUBLISHED RULES ================
Published rule purity on current train:
                                             rule  direction  rows  positives  positive_rate
                  income >= 170537 -> always buys          1   393        393       1.000000
                 income 31004-41970 -> never buys         -1  1257          0       0.000000
                      commute >= 83 -> never buys         -1   186          0       0.000000
30k + no subsidy + concern1/anxious -> never buys         -1  7157          0       0.000000

Rule fold deltas:
+0.000012584 +0.000011202 +0.000010828 +0.000010254 +0.000011900
Mean: 1.1353612870323992e-05
Rules accepted: True
AUC before: 0.946140989060347 | after/fallback: 0.

,candidate,oof_auc,delta_vs_v5
0,rules_experimental,0.946152,0.000054
1,nested_hill,0.946141,0.000042
2,oof_gated_final,0.946141,0.000042
3,team_75hill_25stack,0.946138,0.000039
4,logistic_stack,0.946115,0.000016
5,v5_baseline,0.946099,0.000000


## 18. 跑完后把这些结果发给我

把最后输出截图给我，尤其是：

```text
Nested hill OOF
V5 OOF
Wins vs V5
Nested hill accepted

Logistic stack OOF

Team OOF
Team fold gains vs hill source
Team accepted

Rules accepted

V7 FINAL COMPARISON
```

### 我们最希望看到什么

理想情况：

```text
nested hill > 0.946098745
并且 wins >= 4/5
```

然后：

```text
team > nested hill
并且 team wins >= 4/5
```

如果这两关都通过，那么 `submission_v7_next_candidate.csv` 就会自动指向纯 OOF 证据支持的版本。

### Rules 文件

即使 hard rules 的 OOF 表面上提高，我仍然把它单独保存为：

`submission_v7_rules_experimental.csv`

因为规则本身来自社区对完整训练集的事后发现，不能把它的 OOF gain 和真正 nested meta-CV 的 gain 当成同一种证据。

### 如果 V7 没超过 V5

也不要继续硬调 stack 参数。

那说明高分 notebook 的真正优势主要来自它的 **83 个不同 feature/model views**，而不是 hill climb 算法本身。下一步应该寻找一个真正低相关的新 engine（例如 BlamerX-style window encoding / RealMLP / 不同 feature view），再加入这个 V7 框架，而不是继续重复训练高度相似的 GBDT。


## 19. 为什么 V8 先升级 XGB，而不是立刻换 Transformer

目前公开证据很清楚：

- 你的旧 XGB OOF：约 `0.94593`
- 公开强 XGB 单模：已经能到 Public `0.94640` 左右
- RealMLP 单模公开 Public 大约 `0.94621`

所以**你的 XGB 还有明显未吃到的空间**。

同时，RealMLP / Transformer 的真正价值主要是低相关性，适合在强 GBDT 之后作为 ensemble member，而不是用一个较弱的 NN 直接替换树模型。

因此 V8 的判断标准：

- `strong_rich_te OOF >= 0.94615`：值得继续；
- `>= 0.94620`：明显成功；
- 如果新 XGB + V8 ensemble 仍 <= `0.94616`：停止继续调 GBDT，转 RealMLP。


In [34]:
# ===== V8 configuration =====

V8_FOLDS = 10
V8_SEED = 42

# 两个 feature views。若你的 Mac 跑得太慢，可先只保留 ("strong_rich_te",)
V8_VIEWS = ("strong_rich_te", "strong_digit_freq")

# 第一次保持 None；跑完后可以填结果目录直接复用。
V8_RESULT_DIR = None
# 示例：
# V8_RESULT_DIR = PROJECT / "outputs/v8_strong_xgb_20260930_xxxxxx"

# 公开强 XGB recipe 的核心参数。
# 原公开配置 n_estimators=100000 + early stopping=500；
# 这里仍保留高上限，真正训练轮数由 early stopping 决定。
V8_XGB_PARAMS = dict(
    n_estimators=100000,
    learning_rate=0.01,
    max_depth=4,
    max_leaves=512,
    grow_policy="lossguide",
    min_child_weight=4.532387806880492,
    gamma=3.673225596759869,
    subsample=0.7400402414525654,
    colsample_bytree=0.5695776529558766,
    reg_alpha=0.7523885021652775,
    reg_lambda=0.6189949691705282,
    max_bin=1024,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    early_stopping_rounds=500,
    n_jobs=THREADS,
    verbosity=0,
)

print("V8 folds:", V8_FOLDS)
print("V8 views:", V8_VIEWS)
print("XGB learning rate:", V8_XGB_PARAMS["learning_rate"])
print("XGB grow policy:", V8_XGB_PARAMS["grow_policy"])


V8 folds: 10
V8 views: ('strong_rich_te', 'strong_digit_freq')
XGB learning rate: 0.01
XGB grow policy: lossguide


## 20. V8 fold encoding：rich TE 与 digit/frequency 两种视图

`strong_rich_te` 调用 V5 已经验证过的 nested TE encoder。

`strong_digit_freq` 刻意移除 target encoding，只保留：

- row-local digit decomposition
- 显式 interactions
- fold-local frequency encoding
- fold-local categorical integer mapping

这样可以同时测试两个问题：

1. XGB 参数是不是旧版主要瓶颈；
2. TE 是否让 XGB 和 LGB/Cat 的排序变得过于相似。


In [35]:
def encode_xgb_fold_no_te(train_x, valid_x, test_x):
    """Fold-local frequency encoding, without target encoding."""
    a, b, c = [
        d.copy().reset_index(drop=True)
        for d in (train_x, valid_x, test_x)
    ]

    original_cols = train_x.columns.tolist()
    cat_cols = [col for col in original_cols if _is_cat(a[col])]
    numeric_exact = [col for col in NUMERIC if col in a.columns]
    freq_keys = list(dict.fromkeys(cat_cols + numeric_exact))

    n = len(a)

    # Frequency is learned only from outer training fold.
    for col in freq_keys:
        counts = a[col].value_counts(dropna=False)

        a[f"{col}_freq"] = (
            (a[col].map(counts) - 1).clip(lower=0)
            / max(n - 1, 1)
        ).astype(np.float32)

        b[f"{col}_freq"] = (
            b[col].map(counts).fillna(0) / n
        ).astype(np.float32)

        c[f"{col}_freq"] = (
            c[col].map(counts).fillna(0) / n
        ).astype(np.float32)

    # Learn raw-category integer maps from training fold only.
    for col in cat_cols:
        categories = pd.Index(pd.unique(a[col].astype(str)))
        mapping = {
            value: idx
            for idx, value in enumerate(categories)
        }

        for frame in (a, b, c):
            frame[col] = (
                frame[col].astype(str)
                .map(mapping)
                .fillna(-1)
                .astype(np.int32)
            )

    for frame in (a, b, c):
        for col in frame.columns:
            if pd.api.types.is_bool_dtype(frame[col]):
                frame[col] = frame[col].astype(np.int8)

        bad = frame.select_dtypes(
            include=["object", "string", "category"]
        ).columns.tolist()
        require(
            not bad,
            f"Non-numeric V8 XGB columns remain: {bad}",
        )

        arr = frame.to_numpy(dtype=np.float64, copy=False)
        require(
            np.isfinite(arr).all(),
            "Non-finite V8 XGB feature matrix",
        )

    require(
        a.columns.tolist() == b.columns.tolist() == c.columns.tolist(),
        "V8 XGB train/valid/test columns differ",
    )

    return a, b, c


def encode_v8_view(view, train_x, train_y, valid_x, test_x, seed):
    if view == "strong_rich_te":
        return encode_xgb_fold(
            train_x,
            train_y,
            valid_x,
            test_x,
            seed=seed,
        )

    if view == "strong_digit_freq":
        return encode_xgb_fold_no_te(
            train_x,
            valid_x,
            test_x,
        )

    raise ValueError(f"Unknown V8 view: {view}")


## 21. 训练 Strong XGB 两个视图

这一步是 V8 唯一真正耗时的部分。

每个 view 都：

- 10-fold Stratified CV
- seed=42
- 每折 early stopping
- 保存 model / OOF / test prediction / fold score

### 本地 Mac 提示

这套参数比 V5 重。

如果你只想先快速验证方向，把：

```python
V8_VIEWS = ("strong_rich_te",)
```

先只跑一个。

如果你在 Kaggle GPU 上跑，可以在参数中加：

```python
device="cuda"
```

Mac 的 XGBoost 不使用 MPS，所以本地仍然是 CPU。


In [36]:
def run_v8_strong_xgb(project, result_dir=None):
    try:
        import xgboost as xgb
    except ImportError as error:
        raise RuntimeError(
            "缺少 xgboost。请先安装：%pip install -U xgboost"
        ) from error

    train, test, y, _ = load_project(project)
    hashes_before = input_hashes(project)

    x_all = add_xgb_row_features(train)
    x_test = add_xgb_row_features(test)

    require(
        x_all.columns.tolist() == x_test.columns.tolist(),
        "V8 row feature columns differ",
    )

    folds10 = list(
        StratifiedKFold(
            V8_FOLDS,
            shuffle=True,
            random_state=V8_SEED,
        ).split(train, y)
    )

    if result_dir is not None:
        out = Path(result_dir)
        config_path = out / "config.json"
        if not config_path.is_file():
            raise FileNotFoundError(config_path)

        config = json.loads(config_path.read_text())
        require(
            config.get("input_sha256") == hashes_before,
            "V8 saved-data hashes differ",
        )

        result = {
            "dir": out,
            "folds": folds10,
            "views": {},
        }

        for view in config["views"]:
            oof_frame = align_csv(
                out / f"oof_{view}.csv",
                train.id,
                ["target", "prediction"],
            )
            sub_frame = align_csv(
                out / f"submission_{view}.csv",
                test.id,
                [TARGET],
            )

            oof = probabilities(
                oof_frame["prediction"],
                len(train),
                f"{view} OOF",
            )
            test_pred = probabilities(
                sub_frame[TARGET],
                len(test),
                f"{view} test",
            )

            auc = float(roc_auc_score(y, oof))

            result["views"][view] = {
                "oof": oof,
                "test": test_pred,
                "auc": auc,
            }

            print(f"复用 {view}: {auc:.9f}")

        return result

    out = (
        Path(project)
        / "outputs"
        / ("v8_strong_xgb_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    )
    out.mkdir(parents=True, exist_ok=False)

    config = {
        "folds": V8_FOLDS,
        "seed": V8_SEED,
        "views": list(V8_VIEWS),
        "params": V8_XGB_PARAMS,
        "input_sha256": hashes_before,
        "note": (
            "Strong lossguide XGB recipe inspired by public S6E9 "
            "single-model experiments; two feature views are compared."
        ),
    }

    (out / "config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    result = {
        "dir": out,
        "folds": folds10,
        "views": {},
    }

    for view in V8_VIEWS:
        print(
            f"\n================ V8 XGB VIEW: {view} ================",
            flush=True,
        )

        oof = np.full(len(train), np.nan, dtype=np.float64)
        coverage = np.zeros(len(train), dtype=np.int8)
        test_sum = np.zeros(len(test), dtype=np.float64)
        fold_rows = []

        for fold, (ti, vi) in enumerate(folds10, start=1):
            print(
                f"\n---------- {view} | Fold {fold}/{V8_FOLDS} ----------",
                flush=True,
            )

            a0 = x_all.iloc[ti]
            b0 = x_all.iloc[vi]

            a, b, c = encode_v8_view(
                view,
                a0,
                y.iloc[ti],
                b0,
                x_test,
                seed=8000 + fold,
            )

            params = dict(V8_XGB_PARAMS)
            params["random_state"] = V8_SEED + fold

            model = xgb.XGBClassifier(**params)

            model.fit(
                a,
                y.iloc[ti],
                eval_set=[(b, y.iloc[vi])],
                verbose=250,
            )

            vp = model.predict_proba(b)[:, 1]
            tp = model.predict_proba(c)[:, 1]

            require(
                np.isfinite(vp).all()
                and np.isfinite(tp).all(),
                f"{view} fold {fold}: invalid predictions",
            )

            oof[vi] = vp
            coverage[vi] += 1
            test_sum += tp

            fold_auc = float(
                roc_auc_score(y.iloc[vi], vp)
            )
            best_iteration = getattr(
                model,
                "best_iteration",
                None,
            )

            fold_rows.append({
                "view": view,
                "fold": fold,
                "auc": fold_auc,
                "best_iteration": (
                    None
                    if best_iteration is None
                    else int(best_iteration)
                ),
                "n_features": int(a.shape[1]),
            })

            model.save_model(
                str(out / f"{view}_fold{fold}.json")
            )

            print(
                f"Fold {fold} AUC: {fold_auc:.9f} | "
                f"best_iteration={best_iteration}"
            )

            del a0, b0, a, b, c, model, vp, tp
            gc.collect()

        require(
            (coverage == 1).all(),
            f"{view}: OOF coverage error",
        )
        require(
            np.isfinite(oof).all(),
            f"{view}: missing OOF predictions",
        )
        require(
            input_hashes(project) == hashes_before,
            "Input files changed during V8 training",
        )

        test_pred = test_sum / V8_FOLDS
        total_auc = float(roc_auc_score(y, oof))

        pd.DataFrame(fold_rows).to_csv(
            out / f"fold_scores_{view}.csv",
            index=False,
        )

        pd.DataFrame({
            "id": train.id.to_numpy(),
            "target": y.to_numpy(),
            "prediction": oof,
        }).to_csv(
            out / f"oof_{view}.csv",
            index=False,
        )

        sub = pd.DataFrame({
            "id": test.id.to_numpy(),
            TARGET: test_pred,
        })

        sample_path = Path(project) / "data/sample_submission.csv"
        if sample_path.exists():
            sample_ids = pd.read_csv(
                sample_path,
                usecols=["id"],
            )
            sub = sample_ids.merge(
                sub,
                on="id",
                how="left",
                validate="one_to_one",
                sort=False,
            )

        probabilities(
            sub[TARGET],
            len(test),
            f"{view} submission",
        )
        sub.to_csv(
            out / f"submission_{view}.csv",
            index=False,
        )

        result["views"][view] = {
            "oof": oof,
            "test": test_pred,
            "auc": total_auc,
        }

        print(f"\n{view} OOF AUC: {total_auc:.9f}")

    print("\nV8 strong XGB directory:", out)
    return result


if V8_RESULT_DIR is None:
    v8_xgb = run_v8_strong_xgb(PROJECT)
    V8_RESULT_DIR = v8_xgb["dir"]
else:
    v8_xgb = run_v8_strong_xgb(
        PROJECT,
        V8_RESULT_DIR,
    )



================ V8 XGB VIEW: strong_rich_te ================

---------- strong_rich_te | Fold 1/10 ----------
[0]	validation_0-auc:0.92787
[250]	validation_0-auc:0.94202
[500]	validation_0-auc:0.94386
[750]	validation_0-auc:0.94443


KeyboardInterrupt: 

## 22. 先判断新模型本身是否真的更强、更不同

这里同时看：

- OOF AUC
- vs V5 baseline 的 Spearman
- vs old XGB 的 Spearman

最理想的是：

```text
new XGB OOF  >= 0.94620
corr vs V5   < 0.996
```

如果 AUC 更高、相关性又不完全相同，它才是 V8 ensemble 真正需要的新 engine。


In [ ]:
def v8_model_diagnostics(
    y,
    garage_v7,
    old_xgb,
    v8_xgb,
):
    rows = []

    v5 = garage_v7["v5_oof"]
    old_xgb_oof = old_xgb["oof"]

    rows.append({
        "model": "v5_baseline",
        "oof_auc": float(roc_auc_score(y, v5)),
        "spearman_vs_v5": 1.0,
        "spearman_vs_old_xgb": float(
            pd.Series(v5).corr(
                pd.Series(old_xgb_oof),
                method="spearman",
            )
        ),
    })

    rows.append({
        "model": "old_xgb",
        "oof_auc": float(roc_auc_score(y, old_xgb_oof)),
        "spearman_vs_v5": float(
            pd.Series(old_xgb_oof).corr(
                pd.Series(v5),
                method="spearman",
            )
        ),
        "spearman_vs_old_xgb": 1.0,
    })

    for view, info in v8_xgb["views"].items():
        pred = info["oof"]

        rows.append({
            "model": view,
            "oof_auc": float(roc_auc_score(y, pred)),
            "spearman_vs_v5": float(
                pd.Series(pred).corr(
                    pd.Series(v5),
                    method="spearman",
                )
            ),
            "spearman_vs_old_xgb": float(
                pd.Series(pred).corr(
                    pd.Series(old_xgb_oof),
                    method="spearman",
                )
            ),
        })

    table = (
        pd.DataFrame(rows)
        .sort_values("oof_auc", ascending=False)
        .reset_index(drop=True)
    )

    print(
        table.to_string(
            index=False,
            float_format=lambda v: f"{v:.9f}",
        )
    )

    return table


## 23. 把 Strong XGB 加回 V7 garage，重新做 nested ensemble

这一步不再手调：

`XGB 40% / Current 60%`

而是把：

- CatBoost
- LightGBM
- old XGB
- strong_rich_te
- strong_digit_freq
- 一些透明的 rank/prob blend

统一变成 probit engines，再让 **nested hill** 选择。

这样我们可以直接回答：

> 新 XGB 是只提高单模，还是也真正提高最终 ensemble？


In [37]:
def build_v8_garage(
    old,
    new,
    old_xgb,
    v8_xgb,
):
    cat_oof, cat_test = old["oof"], old["test"]
    lgb_oof, lgb_test = new["oof"], new["test"]
    old_xgb_oof, old_xgb_test = old_xgb["oof"], old_xgb["test"]

    current_oof = 0.5 * cat_oof + 0.5 * lgb_oof
    current_test = 0.5 * cat_test + 0.5 * lgb_test

    v5_oof = (
        0.5 * v7_rank01(old_xgb_oof)
        + 0.5 * v7_rank01(current_oof)
    )
    v5_test = (
        0.5 * v7_rank01(old_xgb_test)
        + 0.5 * v7_rank01(current_test)
    )

    engines_oof = {
        "catboost": cat_oof,
        "lightgbm": lgb_oof,
        "old_xgboost": old_xgb_oof,
        "current_cat_lgb": current_oof,
        "v5_baseline": v5_oof,
    }

    engines_test = {
        "catboost": cat_test,
        "lightgbm": lgb_test,
        "old_xgboost": old_xgb_test,
        "current_cat_lgb": current_test,
        "v5_baseline": v5_test,
    }

    # Add new strong XGB views and direct rank blends with current.
    for view, info in v8_xgb["views"].items():
        engines_oof[view] = info["oof"]
        engines_test[view] = info["test"]

        engines_oof[f"{view}__current_rank50"] = (
            0.5 * v7_rank01(info["oof"])
            + 0.5 * v7_rank01(current_oof)
        )
        engines_test[f"{view}__current_rank50"] = (
            0.5 * v7_rank01(info["test"])
            + 0.5 * v7_rank01(current_test)
        )

    # If both V8 views exist, add their equal-rank model.
    if len(v8_xgb["views"]) >= 2:
        view_names = list(v8_xgb["views"])
        engines_oof["v8_views_equal_rank"] = np.mean(
            [
                v7_rank01(v8_xgb["views"][v]["oof"])
                for v in view_names
            ],
            axis=0,
        )
        engines_test["v8_views_equal_rank"] = np.mean(
            [
                v7_rank01(v8_xgb["views"][v]["test"])
                for v in view_names
            ],
            axis=0,
        )

    names = list(engines_oof)

    P_OOF = np.column_stack([
        v7_probit(engines_oof[n])
        for n in names
    ])
    P_TEST = np.column_stack([
        v7_probit(engines_test[n])
        for n in names
    ])

    return {
        "names": names,
        "raw_oof": engines_oof,
        "raw_test": engines_test,
        "P_OOF": P_OOF,
        "P_TEST": P_TEST,
        "v5_oof": v5_oof,
        "v5_test": v5_test,
    }


def run_v8_final_ensemble(project, v8_xgb):
    train, test, y, folds5 = load_project(project)

    old = read_run(
        BASELINE_DIR,
        "catboost",
        train,
        test,
        y,
        folds5,
        expected_auc=EXPECTED_BASELINE_AUC,
    )

    new = read_run(
        Path(LGB_RESULT_DIR),
        "lightgbm",
        train,
        test,
        y,
        folds5,
    )

    old_xgb = run_xgb10(
        project,
        XGB_RESULT_DIR,
    )

    garage_v7 = build_v7_engines(
        old,
        new,
        old_xgb,
    )

    diagnostics = v8_model_diagnostics(
        y,
        garage_v7,
        old_xgb,
        v8_xgb,
    )

    garage = build_v8_garage(
        old,
        new,
        old_xgb,
        v8_xgb,
    )

    print("\n================ V8 ENGINE GARAGE ================")
    single, fp = v7_engine_diagnostics(
        garage,
        y,
    )

    print("\n================ V8 NESTED HILL ================")
    hill = run_v7_nested_hill(
        garage,
        y,
    )

    print("\n================ V8 STACK + TEAM ================")
    stack = run_v7_stack_and_team(
        garage,
        y,
        hill,
    )

    # Full-data test predictions.
    hill_test = garage["P_TEST"] @ hill["w_full"]
    stack_test = stack["full_stack_model"].decision_function(
        garage["P_TEST"]
    )

    if hill["accepted"]:
        hill_source_test = hill_test
    else:
        hill_source_test = garage["v5_test"]

    team_test = (
        TEAM_HILL_WEIGHT * v7_zscore(hill_source_test)
        + TEAM_STACK_WEIGHT * v7_zscore(stack_test)
    )

    # OOF-only gate.
    if stack["team_accepted"]:
        final_oof = stack["team_oof"]
        final_test = team_test
        final_name = "v8_team"
    elif hill["accepted"]:
        final_oof = hill["oof"]
        final_test = hill_test
        final_name = "v8_nested_hill"
    else:
        # Fall back to the best individual V8 engine if it beats V5.
        all_individual = {
            "v5": garage["v5_oof"],
            **{
                view: info["oof"]
                for view, info in v8_xgb["views"].items()
            },
        }
        best_individual = max(
            all_individual,
            key=lambda k: roc_auc_score(
                y,
                all_individual[k],
            ),
        )

        if best_individual == "v5":
            final_oof = garage["v5_oof"]
            final_test = garage["v5_test"]
            final_name = "v5_fallback"
        else:
            final_oof = v8_xgb["views"][best_individual]["oof"]
            final_test = v8_xgb["views"][best_individual]["test"]
            final_name = best_individual

    out = (
        Path(project)
        / "outputs"
        / ("v8_final_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    )
    out.mkdir(parents=True, exist_ok=False)

    save_v7_submission(
        project,
        test,
        final_test,
        out / "submission_v8_next_candidate.csv",
    )

    # Also save each strong-XGB single submission.
    for view, info in v8_xgb["views"].items():
        save_v7_submission(
            project,
            test,
            info["test"],
            out / f"submission_v8_{view}.csv",
        )

    comparison_rows = [
        {
            "candidate": "v5_baseline",
            "oof_auc": float(
                roc_auc_score(y, garage["v5_oof"])
            ),
        },
        {
            "candidate": "v8_nested_hill",
            "oof_auc": hill["auc"],
        },
        {
            "candidate": "v8_logistic_stack",
            "oof_auc": stack["stack_auc"],
        },
        {
            "candidate": "v8_team",
            "oof_auc": stack["team_auc"],
        },
        {
            "candidate": f"FINAL::{final_name}",
            "oof_auc": float(
                roc_auc_score(y, final_oof)
            ),
        },
    ]

    for view, info in v8_xgb["views"].items():
        comparison_rows.append({
            "candidate": view,
            "oof_auc": info["auc"],
        })

    comparison = pd.DataFrame(comparison_rows)
    baseline_auc = float(
        roc_auc_score(y, garage["v5_oof"])
    )
    comparison["delta_vs_v5"] = (
        comparison["oof_auc"] - baseline_auc
    )
    comparison = (
        comparison
        .sort_values("oof_auc", ascending=False)
        .reset_index(drop=True)
    )

    diagnostics.to_csv(
        out / "v8_model_diagnostics.csv",
        index=False,
    )
    comparison.to_csv(
        out / "v8_comparison.csv",
        index=False,
    )
    hill["fold_table"].to_csv(
        out / "v8_nested_hill_folds.csv",
        index=False,
    )

    report = {
        "final_name": final_name,
        "final_oof_auc": float(
            roc_auc_score(y, final_oof)
        ),
        "v5_oof_auc": baseline_auc,
        "hill_accepted": hill["accepted"],
        "hill_wins": hill["wins"],
        "team_accepted": stack["team_accepted"],
        "team_wins": stack["team_wins"],
        "strong_xgb": {
            view: info["auc"]
            for view, info in v8_xgb["views"].items()
        },
    }

    (out / "v8_report.json").write_text(
        json.dumps(report, indent=2),
        encoding="utf-8",
    )

    print("\n================ V8 FINAL ================")
    print(
        comparison.to_string(
            index=False,
            float_format=lambda v: f"{v:.9f}",
        )
    )
    print("\nSelected:", final_name)
    print(
        "Submission:",
        out / "submission_v8_next_candidate.csv",
    )
    print("Output directory:", out)

    return {
        "dir": out,
        "comparison": comparison,
        "diagnostics": diagnostics,
        "hill": hill,
        "stack": stack,
        "report": report,
    }


v8_final = run_v8_final_ensemble(
    PROJECT,
    v8_xgb,
)

v8_final["comparison"]


复用 XGBoost： /Users/yw/unsw/Kaggle/outputs/v5_xgb10_20260930_205349_440980
XGB OOF AUC: 0.9459315426788591
            model     oof_auc  spearman_vs_v5  spearman_vs_old_xgb
      v5_baseline 0.946098745     1.000000000          0.999088562
   strong_rich_te 0.945998429     0.998514749          0.998273536
          old_xgb 0.945931543     0.999088562          1.000000000
strong_digit_freq 0.944402544     0.994722348          0.993056260

================ V8 ENGINE GARAGE ================
Single-engine OOF AUC:
                           engine     oof_auc
   strong_rich_te__current_rank50 0.946112963
                      v5_baseline 0.946098745
                   strong_rich_te 0.945998429
                  current_cat_lgb 0.945986243
                         lightgbm 0.945946381
                      old_xgboost 0.945931543
              v8_views_equal_rank 0.945759391
strong_digit_freq__current_rank50 0.945620865
                         catboost 0.945618233
                strong_d

/opt/miniconda3/envs/dl/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/miniconda3/envs/dl/lib/python3.11/s

Logistic stack OOF: 0.9461502612961652
Hill source: accepted_nested_hill | AUC: 0.9461777396026639
Team OOF: 0.9461743820017173
Team fold gains vs hill source: -0.000002269 -0.000004222 -0.000004713 -0.000001476 -0.000003167
Team wins: 0/5
Team accepted: False

================ V8 FINAL ================
            candidate     oof_auc  delta_vs_v5
       v8_nested_hill 0.946177740  0.000078994
FINAL::v8_nested_hill 0.946177740  0.000078994
              v8_team 0.946174382  0.000075637
    v8_logistic_stack 0.946150261  0.000051516
          v5_baseline 0.946098745  0.000000000
       strong_rich_te 0.945998429 -0.000100316
    strong_digit_freq 0.944402544 -0.001696201

Selected: v8_nested_hill
Submission: /Users/yw/unsw/Kaggle/outputs/v8_final_20260930_234111_766439/submission_v8_next_candidate.csv
Output directory: /Users/yw/unsw/Kaggle/outputs/v8_final_20260930_234111_766439


,candidate,oof_auc,delta_vs_v5
0,v8_nested_hill,0.946178,0.000079
1,FINAL::v8_nested_hill,0.946178,0.000079
2,v8_team,0.946174,0.000076
3,v8_logistic_stack,0.946150,0.000052
4,v5_baseline,0.946099,0.000000
5,strong_rich_te,0.945998,-0.000100
6,strong_digit_freq,0.944403,-0.001696


## 24. 怎么决定下一步：继续 XGB，还是换 RealMLP

跑完只看这几个数字：

```text
strong_rich_te OOF
strong_digit_freq OOF
spearman_vs_v5

V8 nested hill OOF
V8 team OOF
V8 FINAL
```

### 情况 A：Strong XGB 到 `0.94620+`

说明我们之前主要是 XGB recipe 太弱。

继续用：

`submission_v8_next_candidate.csv`

并且下一步只需小范围 multi-seed。

### 情况 B：单 XGB `0.94610~0.94620`，但 V8 ensemble > `0.94618`

说明新 feature view 有 ensemble 价值，也可以继续。

### 情况 C：两个 XGB 都 < `0.94610`，V8 也几乎不涨

这时就**不要再训练第四个 GBDT**。

下一步换：

**RealMLP / Transformer**

因为公开 RealMLP 单模约 `0.94621`，而公开讨论有人测到 neural/transformer 模型和 GBDT blend 的 rank correlation 约 `0.991`，明显低于很多 GBDT 之间的 `0.992–0.999`。

那时它的价值不是单模必须更强，而是给 ensemble 提供新的排序信息。
